# Phase 2 · 13단원 — ML 파이프라인

> **노트북에서는 잘 되던 게 한 달 뒤 다른 사람이 재학습하면 다른 결과가 나옵니다.**

- 원문: <https://aiengineeringfromscratch.com/lesson?path=phases/02-ml-fundamentals/13-ml-pipelines>
- 예상 시간: 90분 · 선수 지식: Phase 2 1~12단원 (특히 8단원 특징 공학, 9단원 누출)

---

## 📌 출처 표시 규칙

> 📗 **원문** — 커리큘럼에 있는 내용입니다.

> ✨ **추가** — 원문에는 없고, 제가 넣은 비유·예시·실험입니다.

---

## 🎯 이 단원에서 배우는 것

> 📗 **원문** — "Learning Objectives"

1. **커스텀 변환기와 파이프라인**을 밑바닥부터 만들어, `fit`/`transform` 분리가 왜 중요한지 봅니다.
2. **`ColumnTransformer`** 로 숫자·범주 열에 다른 전처리를 적용합니다.
3. **데이터 누출**을 의도적으로 일으키고, 파이프라인이 그걸 어떻게 막는지 측정합니다.
4. **실험 추적(MLflow)·모델 버전 관리·재현성**을 실제로 돌려 봅니다.

### 🔑 Phase 2의 마지막 단원이고, 앞의 전부를 묶는 단원입니다

| 앞 단원에서 배운 것 | 이 단원에서 하는 일 |
| --- | --- |
| 8단원 — 결측 채우기, 스케일링, 인코딩 | **파이프라인 단계로** 조립 |
| 9단원 2절 — 데이터 누출 | **구조적으로 불가능하게** 만들기 |
| 9단원 3절 — 교차검증 | 파이프라인을 **겹마다 새로 fit** |
| 12단원 — 하이퍼파라미터 탐색 | 파이프라인 **전체를** 탐색 대상으로 |

> 📗 원문: "These are not hypothetical. They are the most common reasons ML systems fail
> in production."
> (가정이 아닙니다. **프로덕션에서 ML 시스템이 실패하는 가장 흔한 이유들**입니다.)

---

## 📖 목차

| # | 내용 | 출처 |
| --- | --- | --- |
| 0 | 노트북은 되는데 프로덕션은 깨집니다 | 📗 원문 |
| 1 | 파이프라인이란 | 📗 원문 |
| 2 | ⚠️ 데이터 누출 — 조용한 살인자 | 📗 원문 + ✨ 실험 |
| 3 | 🔨 변환기와 파이프라인 밑바닥부터 | 📗 원문 |
| 4 | 🔨 sklearn Pipeline | 📗 원문 + ✨ 추적 실험 |
| 5 | 🔨 ColumnTransformer | 📗 원문 + ✨ 미지 범주 실험 |
| 6 | 🔑 교차검증 + 파이프라인 | 📗 원문 + ✨ 측정 |
| 7 | 직렬화 — 학습/서빙 불일치 막기 | 📗 원문 + ✨ 실험 |
| 8 | 실험 추적 (MLflow) | 📗 원문 + ✨ 실행 |
| 9 | 재현성 — 네 가지 조건 | 📗 원문 + ✨ 실험 |
| 10 | 노트북에서 프로덕션으로 | 📗 원문 |
| 11 | 🎒 Ship It — 파이프라인 체크리스트 | 📗 원문 `outputs/` |
| 12 | 📝 자가 점검 퀴즈 (5문항) | 📗 원문 `quiz.json` |
| 13 | 용어 사전 · 연습문제 | 📗 원문 |

---

## 🗺️ 이 단원은 이렇게 이어집니다

> ✨ **추가** — 절이 왜 이 순서인지 먼저 짚고 갑니다. 하나가 다음 하나를 불러옵니다.

**0절**에서 **실제로 일어나는 장애 네 가지**를 봅니다. 노트북에서는 다 잘 됐는데
한 달 뒤 깨지는 이유들이죠. **1절**에서 그 네 가지를 한 번에 해결하는 물건을 소개합니다 — **파이프라인**.

그런데 "파이프라인을 쓰세요"로 끝내면 **왜 되는지 모릅니다.** 그래서 순서를 이렇게 잡습니다.

- **2절 ⚠️** — 먼저 **문제를 직접 일으킵니다.** 나누기 전에 스케일러를 `fit` 하면 어떻게 되나.
  9단원 2절에서 맛봤는데, 여기서는 **y를 보는 전처리**까지 포함해 제대로 측정합니다.
- **3절 🔨** — 파이프라인을 **직접 만듭니다.** 10줄이면 됩니다.
  만들어 보면 **`fit_transform` 과 `transform` 의 분리가 전부**라는 걸 알게 됩니다.
- **4절** — sklearn `Pipeline` 으로 같은 일을 합니다. 그리고 **내부에서 무슨 메서드가
  어떤 순서로 불리는지** 추적해서 찍어 봅니다.
- **5절** — 실제 데이터는 숫자·범주가 섞여 있습니다. **`ColumnTransformer`** 가 그걸 라우팅합니다.
  그리고 **프로덕션에서 가장 흔한 사고** — 처음 보는 범주 값 — 를 재현합니다.
- **6절 🔑** — 9단원 3절의 교차검증과 합칩니다. 파이프라인을 쓰면 **겹마다 전처리가 새로 fit** 됩니다.
  2절의 누출이 **구조적으로 불가능해집니다.**

여기까지가 "학습 때 올바르게". 이제 **"배포하고 나서도 올바르게"**입니다.

- **7절** — 파이프라인을 **파일로 저장**합니다. 전처리와 모델이 **한 덩어리**로 묶여야
  학습/서빙 불일치가 안 생깁니다.
- **8절** — 실험이 수십 개가 되면 **뭘 어떻게 돌렸는지 기억 못 합니다.** MLflow로 기록합니다.
- **9절** — 같은 코드가 **같은 결과**를 내게 만드는 네 조건을 확인합니다.
- **10절** — 노트북에서 프로덕션까지의 8단계 경로.

### 한 줄로 꿰면

> **프로덕션이 깨지는 이유(0) → 파이프라인이 답(1) → 문제를 직접 일으키고(2)
> → 직접 만들어 원리를 알고(3) → sklearn으로(4) → 열마다 다르게(5) → 교차검증과 합치고(6)
> → 파일로 묶어 배포하고(7) → 기록하고(8) → 재현 가능하게(9) → 프로덕션으로(10)**

In [1]:
# ── 준비물 ──────────────────────────────────────────────────────
import os
import json as _json
import random
import tempfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)

available = {f.name for f in fm.fontManager.ttflist}
for cand in ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic",
             "NanumBarunGothicOTF", "Malgun Gothic", "Noto Sans CJK KR"]:
    if cand in available:
        matplotlib.rcParams["font.family"] = ["DejaVu Sans", cand]
        break
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["figure.dpi"] = 110

# 이 노트북이 만드는 파일들을 한곳에 모읍니다
작업방 = tempfile.mkdtemp(prefix="pipeline-13-")
print(f"준비 끝! (임시 작업 폴더: {작업방})")

준비 끝! (임시 작업 폴더: /var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2)


## 0️⃣ 노트북은 되는데 프로덕션은 깨집니다

> 📗 **원문** — "The Problem"

> 데이터를 읽고, 결측값을 중앙값으로 채우고, 특징을 스케일링하고, 모델을 학습시켜
> 정확도를 찍는 노트북이 있습니다. **잘 됩니다.** 배포합니다.
>
> **한 달 뒤**, 누군가 모델을 재학습했는데 **다른 결과**가 나옵니다.

### 📗 네 가지 실제 장애

| # | 무슨 일이 | 왜 |
| --- | --- | --- |
| 1 | 중앙값을 **테스트 데이터까지 포함해** 계산했습니다 | **데이터 누출** (2절) |
| 2 | 스케일링 파라미터를 **저장하지 않아** 추론 때 다른 통계를 씁니다 | 학습/서빙 불일치 (7절) |
| 3 | 특징 공학 코드를 학습용·서빙용으로 **복사했는데 사본이 갈라졌습니다** | 한 객체로 묶지 않아서 (7절) |
| 4 | 범주형 열에 **프로덕션에서 처음 보는 값**이 등장했습니다 | 인코더가 터집니다 (5절) |

### 🧸 비유 — 레시피 없이 요리하기

요리를 잘합니다. 손맛으로 소금을 넣고, 눈대중으로 익힙니다. **맛있습니다.**

그런데 **다른 사람에게 그 요리를 맡기면?** 맛이 달라집니다.
**한 달 뒤 내가 다시 해도** 달라집니다. 소금을 얼마나 넣었는지 기억 못 하니까요.

**파이프라인은 레시피를 글로 적어 두는 것**입니다.
그것도 **"소금 한 꼬집"이 아니라 "소금 3.2g"** 처럼 정확하게요.

> 📗 원문: "Pipelines solve all of them by packaging every transformation step
> into a single, ordered, reproducible object."
> (파이프라인은 **모든 변환 단계를 하나의, 순서가 있는, 재현 가능한 객체로** 포장해서
> 이 문제들을 전부 해결합니다.)

In [2]:
# ✨ 추가 — 이 단원에서 계속 쓸 '현실적인' 데이터를 만듭니다
def 고객데이터(n=2000, 시드=0, 새범주=False):
    # 숫자 3개 + 범주 2개 + 결측. 0절의 네 가지 장애를 다 재현할 수 있는 구성입니다
    r = np.random.default_rng(시드)
    도시목록 = ["서울", "부산", "대구", "광주"] + (["세종"] if 새범주 else [])
    요금제목록 = ["베이직", "프로", "엔터프라이즈"]

    도시 = r.choice(도시목록, n)
    요금제 = r.choice(요금제목록, n, p=[0.5, 0.35, 0.15])
    나이 = r.integers(19, 75, n).astype(float)
    소득 = r.lognormal(10.2, 0.9, n)
    사용량 = r.gamma(2.0, 30.0, n)

    도시효과 = {"서울": 0.9, "부산": 0.2, "대구": -0.3, "광주": -0.5, "세종": 0.4}
    요금효과 = {"베이직": -0.6, "프로": 0.3, "엔터프라이즈": 1.1}
    점수 = (np.array([도시효과[c] for c in 도시])
            + np.array([요금효과[p] for p in 요금제])
            + np.where(소득 > np.quantile(소득, 0.7), 0.8, -0.2)
            + np.where((나이 > 30) & (나이 < 55), 0.5, -0.3)
            + 사용량 / 120.0
            + r.normal(0, 0.7, n))
    이탈 = (점수 < np.median(점수)).astype(int)         # 목표: 이탈 여부

    나이[r.random(n) < 0.12] = np.nan                   # 결측 12%
    df = pd.DataFrame({"도시": 도시, "요금제": 요금제, "나이": 나이,
                       "소득": 소득, "사용량": 사용량})
    return df, 이탈

df, y = 고객데이터()
숫자열 = ["나이", "소득", "사용량"]
범주열 = ["도시", "요금제"]

print(f"고객 데이터: {len(df)}행 × {len(df.columns)}열\n")
print(df.head(4).to_string(index=False))
print(f"\n{'열':<10}{'자료형':<12}{'결측':>8}{'고유값':>9}")
print("-" * 42)
for c in df.columns:
    print(f"{c:<9}{str(df[c].dtype):<12}{int(df[c].isna().sum()):>8}{df[c].nunique():>9}")
print(f"\n목표(이탈) 비율: {y.mean():.1%}")
print("\n=> 이 데이터로는 **그냥 모델에 넣을 수 없습니다.**")
print("   '도시'는 글자고, '나이'에는 빈칸이 있고, 세 숫자 열의 스케일이 제각각입니다.")
print("   8단원에서 이걸 하나씩 손으로 고쳤죠. 이번엔 **파이프라인으로 묶습니다.**")

고객 데이터: 2000행 × 5열

도시 요금제   나이           소득        사용량
광주 베이직 68.0  7282.714087  40.514065
대구  프로 28.0 28946.274633 170.976972
대구 베이직 49.0 13920.398964  34.888849
부산 베이직 63.0  9297.735791  33.592835

열         자료형               결측      고유값
------------------------------------------
도시       object             0        4
요금제      object             0        3
나이       float64          248       56
소득       float64            0     2000
사용량      float64            0     2000

목표(이탈) 비율: 50.0%

=> 이 데이터로는 **그냥 모델에 넣을 수 없습니다.**
   '도시'는 글자고, '나이'에는 빈칸이 있고, 세 숫자 열의 스케일이 제각각입니다.
   8단원에서 이걸 하나씩 손으로 고쳤죠. 이번엔 **파이프라인으로 묶습니다.**


---

## 1️⃣ 파이프라인이란

> 📗 **원문** — "What a Pipeline Is"
>
> 🔗 **앞 절과 이어서** — 0절의 네 가지 장애를 한 번에 해결하는 물건입니다.

> 📗 원문: "A pipeline is an ordered sequence of data transformations followed by a model."

```
날데이터 → 결측 채우기 → 숫자 스케일링 → 범주 인코딩 → 모델 → 예측
```

각 단계가 **앞 단계의 출력을 입력으로** 받습니다. 전체를 **훈련 데이터로 한 번 fit** 하고,
추론 때는 **같은 fit된 파이프라인**이 새 데이터를 변환해 예측을 냅니다.

### 📗 파이프라인이 보장하는 네 가지

| 보장 | 어느 장애를 막나 | 이 노트북 |
| --- | --- | --- |
| 변환은 **훈련 데이터로만 fit** 됩니다 | 0절 장애 ① (누출) | 2·6절 |
| **추론 때 같은 변환**이 적용됩니다 | 0절 장애 ② (불일치) | 4·7절 |
| **전체 객체를 하나로** 직렬화·배포할 수 있습니다 | 0절 장애 ③ (사본 분기) | 7절 |
| 교차검증이 **겹마다 파이프라인을 적용**합니다 | 0절 장애 ① (미묘한 누출) | 6절 |

### 🔑 ⭐ 핵심은 메서드 두 개의 분리입니다

| 메서드 | 하는 일 | 언제 |
| --- | --- | --- |
| **`fit_transform`** | 데이터에서 **통계를 배우고** 변환합니다 | **훈련 데이터에만** |
| **`transform`** | **이미 배운 통계로** 변환만 합니다 | **테스트·추론 데이터에** |

> 📗 원문: "The scaler never sees test data during fitting. **This is the whole point.**"
>
> 3절에서 직접 만들어 보면 **정말 이게 전부**라는 걸 알게 됩니다.

---

## 2️⃣ ⚠️ 데이터 누출 — 조용한 살인자

> 📗 **원문** — "Data Leakage: The Silent Killer"
>
> 🔗 **앞 절과 이어서** — 1절에서 "파이프라인이 누출을 막는다"고 했습니다.
> 그럼 **막지 않으면 얼마나 나빠지나요?** 직접 일으켜 봅니다.
>
> 9단원 2절에서 이미 한 번 봤습니다. 거기서 **중요한 발견**이 있었죠 —
> **스케일러 누출은 효과가 거의 없고, y를 보는 전처리는 효과가 막대합니다.**
> 여기서는 **파이프라인이라는 해법**을 중심으로 다시 봅니다.

### 📗 잘못된 코드 (원문)

```python
X = df.drop("target", axis=1)
y = df["target"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)        # ❌ 전체 데이터로 fit

X_train, X_test = X_scaled[:800], X_scaled[800:]
```

> 스케일러가 **테스트 데이터를 봤습니다.** 평균과 표준편차에 테스트 샘플이 포함됩니다.

### 📗 올바른 코드 (원문)

```python
X_train, X_test = X[:800], X[800:]        # ✅ 먼저 나눕니다

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # transform 만!
```

> 📗 원문: "With a pipeline, you do not need to think about this.
> **The pipeline handles it automatically.**"

### 🧸 비유 — 시험 범위를 시험지 보고 정하기

선생님이 시험 범위를 정할 때 **시험지를 먼저 봤다면?** 학생 점수가 높게 나오겠죠.
그게 실력인가요? 아닙니다.

**누출의 무서운 점은 에러가 안 난다는 것**입니다. 점수만 좋아 보일 뿐이에요.
배포하고 나서야 "왜 실제 성능이 이렇지?" 하게 됩니다.

In [3]:
# ✨ 추가 — 누출을 종류별로 일으켜 크기를 재 봅니다
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

# ⭐ 정보가 '완전히 0'인 데이터. 진짜 실력은 정확히 50%여야 합니다
def 무의미데이터(n행=80, n열=600, 시드=0):
    r = np.random.default_rng(시드)
    return r.normal(size=(n행, n열)), r.integers(0, 2, n행)

X0, y0 = 무의미데이터()
겹5 = StratifiedKFold(5, shuffle=True, random_state=0)

print(f"입력 {X0.shape[1]}개 열 × {X0.shape[0]}행. 입력도 정답도 전부 무작위")
print("=> 어떤 모델도 진짜 실력은 **정확히 50%** 입니다\n")
print(f"{'전처리':<24}{'y를 보나':>10}{'❌ 미리 fit':>13}{'✅ Pipeline':>13}{'부풀림':>10}")
print("-" * 72)

누출표 = []
for 이름, 만들기, y봄 in [
        ("StandardScaler", lambda: StandardScaler(), "아니오"),
        ("SimpleImputer(median)", lambda: SimpleImputer(strategy="median"), "아니오"),
        ("SelectKBest(k=10)", lambda: SelectKBest(f_classif, k=10), "⭐ 예")]:
    미리 = 만들기(); 미리.fit(X0, y0)
    나쁨 = float(np.mean(cross_val_score(LogisticRegression(max_iter=3000),
                                        미리.transform(X0), y0, cv=겹5)))
    좋음 = float(np.mean(cross_val_score(
        Pipeline([("전처리", 만들기()), ("모델", LogisticRegression(max_iter=3000))]),
        X0, y0, cv=겹5)))
    누출표.append((이름, 나쁨, 좋음))
    print(f"{이름:<23}{y봄:>10}{나쁨:>13.1%}{좋음:>13.1%}{나쁨-좋음:>+10.1%}")
print(f"{'(정답)':<59}{0.5:>10.1%}")

y안봄 = [r for r in 누출표 if "Select" not in r[0]]
y봄행 = [r for r in 누출표 if "Select" in r[0]][0]
print(f"\n=> ⚠️ **y를 보느냐가 전부입니다.** (9단원 2절에서 본 그 결론)")
print(f"   · y를 안 보는 전처리: 부풀림 최대 {max(abs(r[1]-r[2]) for r in y안봄):.1%}  (거의 없음)")
print(f"   · y를 보는 전처리  : 부풀림 {y봄행[1]-y봄행[2]:+.1%}  (**막대함**)")
print("\n   그런데 📗 원문은 **스케일러 누출도 하지 말라**고 합니다. 왜일까요?")
print("   ① 효과가 작다고 **0은 아닙니다.** 테스트 분포가 훈련과 다르면 커집니다")
print("   ② 무엇보다, **구분해서 기억할 필요가 없어집니다.**")
print("      Pipeline에 다 넣으면 '이건 y를 보나?' 고민할 일이 사라집니다")
print("\n   💡 이게 파이프라인의 진짜 가치입니다 — **실수할 여지를 없애는 것.**")

입력 600개 열 × 80행. 입력도 정답도 전부 무작위
=> 어떤 모델도 진짜 실력은 **정확히 50%** 입니다

전처리                          y를 보나     ❌ 미리 fit   ✅ Pipeline       부풀림
------------------------------------------------------------------------
StandardScaler                아니오        61.3%        60.0%     +1.3%
SimpleImputer(median)         아니오        60.0%        60.0%     +0.0%
SelectKBest(k=10)             ⭐ 예        80.0%        50.0%    +30.0%
(정답)                                                            50.0%

=> ⚠️ **y를 보느냐가 전부입니다.** (9단원 2절에서 본 그 결론)
   · y를 안 보는 전처리: 부풀림 최대 1.3%  (거의 없음)
   · y를 보는 전처리  : 부풀림 +30.0%  (**막대함**)

   그런데 📗 원문은 **스케일러 누출도 하지 말라**고 합니다. 왜일까요?
   ① 효과가 작다고 **0은 아닙니다.** 테스트 분포가 훈련과 다르면 커집니다
   ② 무엇보다, **구분해서 기억할 필요가 없어집니다.**
      Pipeline에 다 넣으면 '이건 y를 보나?' 고민할 일이 사라집니다

   💡 이게 파이프라인의 진짜 가치입니다 — **실수할 여지를 없애는 것.**


In [4]:
# ✨ 추가 — ⚠️ 누출은 점수를 "부풀리기만" 하지 않습니다. **망가뜨릴 수도** 있습니다
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

def 누출비교(만들기모델, 만들기스케일러, 이상치있음, 시드=0):
    r = np.random.default_rng(시드)
    n = 500
    Xa = r.normal(0, 1, (n, 10)); ya = (Xa[:, :3].sum(1) > 0).astype(int)
    Xb = r.normal(0, 1, (n, 10)); yb = (Xb[:, :3].sum(1) > 0).astype(int)
    if 이상치있음:
        Xb[:5] = 500.0                    # ⭐ 테스트에 극단 이상치 5행 (센서 고장 같은 상황)

    # ❌ 전체(훈련+테스트)로 스케일러 fit
    s1 = 만들기스케일러().fit(np.vstack([Xa, Xb]))
    나쁨 = 만들기모델().fit(s1.transform(Xa), ya).score(s1.transform(Xb), yb)
    # ✅ 훈련만으로 fit
    s2 = 만들기스케일러().fit(Xa)
    좋음 = 만들기모델().fit(s2.transform(Xa), ya).score(s2.transform(Xb), yb)
    return 나쁨, 좋음

print("스케일러는 y를 안 봅니다. 그럼 전체로 fit 해도 괜찮을까요?\n")
print(f"{'모델':<14}{'스케일러':<16}{'테스트 이상치':>12}"
      f"{'❌ 전체 fit':>13}{'✅ 훈련 fit':>13}{'차이':>9}")
print("-" * 80)
방향기록 = []
for 모델이름, 만들기 in [("로지스틱", lambda: LogisticRegression(max_iter=2000)),
                        ("KNN (k=5)", lambda: KNeighborsClassifier(5))]:
    for sc이름, sc in [("StandardScaler", StandardScaler), ("MinMaxScaler", MinMaxScaler)]:
        for 이상치 in [False, True]:
            나쁨, 좋음 = 누출비교(만들기, sc, 이상치)
            방향기록.append((모델이름, sc이름, 이상치, 나쁨, 좋음))
            print(f"{모델이름:<13}{sc이름:<16}{'있음' if 이상치 else '없음':>12}"
                  f"{나쁨:>13.1%}{좋음:>13.1%}{나쁨-좋음:>+9.1%}")

최악 = min(방향기록, key=lambda r: r[3] - r[4])
print(f"\n=> ⭐ **가장 큰 차이: {최악[0]} + {최악[1]} + 이상치 "
      f"{'있음' if 최악[2] else '없음'}**")
print(f"   ❌ {최악[3]:.1%}  vs  ✅ {최악[4]:.1%}   ({최악[3]-최악[4]:+.1%}p)")
print(f"\n   ⚠️ **누출한 쪽이 더 나쁩니다.** '누출하면 점수가 부풀려진다'와 반대죠.")
print("   왜 그럴까요? MinMaxScaler 는 (x − min) / (max − min) 입니다.")
print("   테스트에 500 짜리 이상치가 끼면 max=500 이 되고,")
print("   **정상 데이터 전부가 0 근처로 짓눌립니다.** 신호가 사라지는 거죠.")
print("   8단원 2절에서 '최소-최대는 이상치 하나에 무너진다'를 측정했습니다. 그 일입니다.")

print("\n💡 그래서 정확한 정리는 이렇습니다 — **누출은 방향이 일정하지 않습니다.**")
print("   · y를 **보는** 전처리 (특징 선택, 타깃 인코딩) -> 점수를 **부풀립니다** (앞 셀 +30%)")
print("   · y를 **안 보지만 이상치에 취약한** 전처리 (MinMax) -> 점수를 **망가뜨릴 수도** 있습니다")
print("   · 어느 쪽이든 **틀린 추정**입니다. 배포하면 다른 결과가 나옵니다.")
print("\n   ⭐ 📗 원문이 'Always split first, then preprocess' 라고 **예외 없이** 말하는 이유입니다.")
print("      '이건 y를 안 보니까 괜찮겠지' 라는 판단이 틀릴 수 있으니까요.")
print("      Pipeline 에 다 넣으면 **그 판단 자체를 할 필요가 없습니다.**")

스케일러는 y를 안 봅니다. 그럼 전체로 fit 해도 괜찮을까요?

모델            스케일러                 테스트 이상치     ❌ 전체 fit     ✅ 훈련 fit       차이
--------------------------------------------------------------------------------
로지스틱         StandardScaler            없음        98.6%        98.6%    +0.0%
로지스틱         StandardScaler            있음        89.4%        98.0%    -8.6%
로지스틱         MinMaxScaler              없음        95.4%        95.2%    +0.2%
로지스틱         MinMaxScaler              있음        49.4%        94.6%   -45.2%
KNN (k=5)    StandardScaler            없음        83.2%        82.8%    +0.4%
KNN (k=5)    StandardScaler            있음        82.8%        82.2%    +0.6%
KNN (k=5)    MinMaxScaler              없음        82.8%        84.2%    -1.4%
KNN (k=5)    MinMaxScaler              있음        83.0%        83.6%    -0.6%

=> ⭐ **가장 큰 차이: 로지스틱 + MinMaxScaler + 이상치 있음**
   ❌ 49.4%  vs  ✅ 94.6%   (-45.2%p)

   ⚠️ **누출한 쪽이 더 나쁩니다.** '누출하면 점수가 부풀려진다'와 반대죠.
   왜 그럴까요? MinMaxScaler 는 (x − min) / (max − min) 입니다.

---

## 3️⃣ 🔨 변환기와 파이프라인 밑바닥부터

> 📗 **원문** — "Build It: Step 1, 2"
>
> 🔗 **앞 절과 이어서** — 2절에서 문제를 봤습니다.
> 이제 해법을 **직접 만들어** 봅니다. 놀랄 만큼 짧습니다.

### 📗 변환기의 계약 (원문 Step 1)

변환기는 **세 메서드**만 있으면 됩니다.

| 메서드 | 하는 일 |
| --- | --- |
| `fit(X)` | 데이터에서 **통계를 배워 자기 안에 저장** |
| `transform(X)` | 저장한 통계로 **변환만** |
| `fit_transform(X)` | 둘을 연달아 |

### 📗 파이프라인의 계약 (원문 Step 2)

- `fit`: 마지막 단계 **앞까지는 `fit_transform`**, 마지막 모델은 `fit`
- `predict`: 마지막 단계 **앞까지는 `transform`**, 마지막 모델은 `predict`

### 🔑 ⭐ 여기가 전부입니다

**`fit` 때는 `fit_transform`, `predict` 때는 `transform`.**
이 한 줄 차이가 2절의 누출을 막습니다. 정말 그게 다예요.

In [5]:
# 📗 원문 Step 1 — 커스텀 변환기
class 표준화변환기:
    def __init__(self):
        self.평균 = None
        self.표준편차 = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.평균 = np.nanmean(X, axis=0)
        self.표준편차 = np.nanstd(X, axis=0)
        self.표준편차[self.표준편차 == 0] = 1.0        # 0으로 나눔 방지
        return self

    def transform(self, X):
        if self.평균 is None:
            raise RuntimeError("fit 을 먼저 불러야 합니다")
        return (np.asarray(X, dtype=float) - self.평균) / self.표준편차

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


class 중앙값채우기:
    # ✨ 추가 — 8단원 5절의 '표시 열'까지 넣습니다
    def __init__(self, 표시열=True):
        self.표시열 = 표시열
        self.채울값 = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.채울값 = np.nanmedian(X, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        빈칸 = np.isnan(X)
        채운것 = np.where(빈칸, self.채울값, X)
        return np.hstack([채운것, 빈칸.astype(float)]) if self.표시열 else 채운것

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


# 📗 원문 Step 2 — 파이프라인
class 내파이프라인:
    def __init__(self, 단계들):
        self.단계들 = 단계들        # [(이름, 객체), ...] 마지막이 모델

    def fit(self, X, y=None):
        현재 = X
        for 이름, 단계 in self.단계들[:-1]:
            현재 = 단계.fit_transform(현재, y)      # ⭐ 훈련 때는 fit_transform
        이름, 모델 = self.단계들[-1]
        모델.fit(현재, y)
        return self

    def _변환만(self, X):
        현재 = X
        for 이름, 단계 in self.단계들[:-1]:
            현재 = 단계.transform(현재)             # ⭐ 예측 때는 transform 만
        return 현재

    def predict(self, X):
        return self.단계들[-1][1].predict(self._변환만(X))

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))

print("변환기·파이프라인 준비 완료 — 코드 40줄이 전부입니다")

변환기·파이프라인 준비 완료 — 코드 40줄이 전부입니다


In [6]:
# ✨ 추가 — 직접 만든 것과 sklearn이 같은 답을 내는지
X숫자 = df[숫자열].to_numpy(dtype=float)
Xa, Xb, ya, yb = train_test_split(X숫자, y, test_size=0.3, random_state=0, stratify=y)

내것 = 내파이프라인([("채우기", 중앙값채우기(표시열=False)),
                    ("표준화", 표준화변환기()),
                    ("모델", LogisticRegression(max_iter=2000))]).fit(Xa, ya)
sk것 = Pipeline([("채우기", SimpleImputer(strategy="median")),
                ("표준화", StandardScaler()),
                ("모델", LogisticRegression(max_iter=2000))]).fit(Xa, ya)

print(f"{'구현':<22}{'훈련 정확도':>12}{'테스트 정확도':>14}")
print("-" * 50)
print(f"{'직접 만든 것 (3절)':<20}{내것.score(Xa, ya):>12.1%}{내것.score(Xb, yb):>14.1%}")
print(f"{'sklearn Pipeline':<21}{sk것.score(Xa, ya):>12.1%}{sk것.score(Xb, yb):>14.1%}")
print(f"\n두 예측이 일치하는 비율: {np.mean(내것.predict(Xb) == sk것.predict(Xb)):.1%}")
print("=> 같으면 제대로 만든 겁니다. **40줄로 sklearn Pipeline의 핵심을 재현했습니다.**")

print("\n[⚠️ fit 을 건너뛰면?]")
안배운것 = 표준화변환기()
try:
    안배운것.transform(Xb)
except RuntimeError as e:
    print(f"   RuntimeError: {e}")
print("   => 변환기는 **fit 없이 transform 할 수 없습니다.**")
print("      이 제약이 '테스트 데이터로 fit 하는 실수'를 막아 줍니다.")
print("      sklearn도 `NotFittedError` 로 같은 일을 합니다.")

구현                          훈련 정확도       테스트 정확도
--------------------------------------------------
직접 만든 것 (3절)               63.6%         64.5%
sklearn Pipeline            63.6%         64.5%

두 예측이 일치하는 비율: 100.0%
=> 같으면 제대로 만든 겁니다. **40줄로 sklearn Pipeline의 핵심을 재현했습니다.**

[⚠️ fit 을 건너뛰면?]
   RuntimeError: fit 을 먼저 불러야 합니다
   => 변환기는 **fit 없이 transform 할 수 없습니다.**
      이 제약이 '테스트 데이터로 fit 하는 실수'를 막아 줍니다.
      sklearn도 `NotFittedError` 로 같은 일을 합니다.


---

## 4️⃣ 🔨 sklearn Pipeline

> 📗 **원문** — "sklearn Pipeline"
>
> 🔗 **앞 절과 이어서** — 3절에서 원리를 알았으니, 이제 실무용 도구를 씁니다.

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])
pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)
```

### 📗 `pipe.fit(X_train, y_train)` 을 부르면

1. 스케일러가 `X_train` 에 **`fit_transform`**
2. 모델이 스케일된 `X_train` 에 **`fit`**

### 📗 `pipe.predict(X_test)` 를 부르면

1. 스케일러가 `X_test` 에 **`transform`** (`fit_transform` 아님!)
2. 모델이 스케일된 `X_test` 에 **`predict`**

> 📗 원문: "The scaler never sees test data during fitting. This is the whole point."

### ✨ 정말 그런지 **추적해서 찍어 봅니다**

말로만 들으면 믿기 어렵죠. 메서드가 불릴 때마다 로그를 남기는 변환기를 만들어
**실제 호출 순서**를 눈으로 확인하겠습니다.

In [7]:
# ✨ 추가 — Pipeline 내부에서 어떤 메서드가 언제 불리는지 추적합니다
from sklearn.base import BaseEstimator, TransformerMixin

호출기록 = []

class 추적변환기(BaseEstimator, TransformerMixin):
    # sklearn 규약을 따르는 변환기 (BaseEstimator 를 상속하면 Pipeline 에 들어갑니다)
    def __init__(self, 이름="추적"):
        self.이름 = 이름

    def fit(self, X, y=None):
        호출기록.append(f"{self.이름}.fit(행 {len(X)}개)")
        self.평균_ = np.asarray(X, dtype=float).mean(axis=0)
        return self

    def transform(self, X):
        호출기록.append(f"{self.이름}.transform(행 {len(X)}개)")
        return np.asarray(X, dtype=float) - self.평균_


# ⚠️ 추적변환기에는 결측 처리가 없으니 결측 없는 행만 씁니다
온전 = ~np.isnan(X숫자).any(axis=1)
Xc, Xd, yc, yd = train_test_split(X숫자[온전], y[온전], test_size=0.3,
                                  random_state=0, stratify=y[온전])

추적관 = Pipeline([("A", 추적변환기("A")), ("B", 추적변환기("B")),
                  ("모델", LogisticRegression(max_iter=2000))])

호출기록.clear()
추적관.fit(Xc, yc)
print(f"pipe.fit(훈련 {len(Xc)}행) 을 부르면:")
for i, 줄 in enumerate(호출기록, 1):
    print(f"   {i}. {줄}")

호출기록.clear()
추적관.predict(Xd)
print(f"\npipe.predict(테스트 {len(Xd)}행) 을 부르면:")
for i, 줄 in enumerate(호출기록, 1):
    print(f"   {i}. {줄}")

print("\n=> ⭐ **fit 때는 fit 이 불리고, predict 때는 transform 만 불립니다.**")
print(f"   테스트 {len(Xd)}행에 대해 `fit` 이 **한 번도 안 불렸습니다.**")
print("   📗 원문의 'The scaler never sees test data during fitting' 이 이 로그입니다.")
print("\n   ⚠️ 그리고 `TransformerMixin` 을 상속하면 `fit_transform` 이 공짜로 생깁니다.")
print("      그래서 위 클래스에는 `fit_transform` 을 안 썼는데도 Pipeline 이 씁니다.")

pipe.fit(훈련 1226행) 을 부르면:
   1. A.fit(행 1226개)
   2. A.transform(행 1226개)
   3. B.fit(행 1226개)
   4. B.transform(행 1226개)

pipe.predict(테스트 526행) 을 부르면:
   1. A.transform(행 526개)
   2. B.transform(행 526개)

=> ⭐ **fit 때는 fit 이 불리고, predict 때는 transform 만 불립니다.**
   테스트 526행에 대해 `fit` 이 **한 번도 안 불렸습니다.**
   📗 원문의 'The scaler never sees test data during fitting' 이 이 로그입니다.

   ⚠️ 그리고 `TransformerMixin` 을 상속하면 `fit_transform` 이 공짜로 생깁니다.
      그래서 위 클래스에는 `fit_transform` 을 안 썼는데도 Pipeline 이 씁니다.


---

## 5️⃣ 🔨 ColumnTransformer

> 📗 **원문** — "ColumnTransformer: Different Pipelines for Different Columns"
>
> 🔗 **앞 절과 이어서** — 4절까지는 **모든 열에 같은 처리**를 했습니다.
> 그런데 0절 데이터에는 숫자 3개와 **글자 2개**가 섞여 있습니다.
> 글자에 `StandardScaler` 를 쓸 수는 없죠.

> 📗 원문: "Real datasets have numeric and categorical columns that need different
> preprocessing. `ColumnTransformer` handles this."

```python
numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                             ("encode", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, ["age", "income", "score"]),
    ("cat", categorical_pipe, ["city", "gender", "plan"]),
])
```

### 🧸 비유 — 분리수거

쓰레기를 한 통에 넣으면 안 됩니다. **종이는 종이통, 플라스틱은 플라스틱통.**
`ColumnTransformer` 가 **열마다 올바른 통으로 보내 주는 장치**입니다.
그리고 처리가 끝나면 **다시 하나로 합쳐** 모델에 넘깁니다.

### ⚠️ ⭐ 📗 프로덕션에서 가장 중요한 한 줄

> "The `handle_unknown="ignore"` in OneHotEncoder is **critical for production**.
> When a new category appears (a city the model has never seen), it produces
> a zero vector instead of crashing."

0절 장애 ④가 이것입니다. **아래에서 실제로 터뜨려 봅니다.**

In [8]:
# 📗 원문 — ColumnTransformer 로 열마다 다른 전처리
숫자관 = Pipeline([("채우기", SimpleImputer(strategy="median")),
                  ("스케일", StandardScaler())])
범주관 = Pipeline([("채우기", SimpleImputer(strategy="most_frequent")),
                  ("원핫", OneHotEncoder(handle_unknown="ignore"))])
전처리 = ColumnTransformer([("숫자", 숫자관, 숫자열),
                           ("범주", 범주관, 범주열)])

전체관 = Pipeline([("전처리", 전처리),
                  ("모델", LogisticRegression(max_iter=3000))])

df훈련, df테스트, y훈련, y테스트 = train_test_split(df, y, test_size=0.3,
                                                 random_state=0, stratify=y)
전체관.fit(df훈련, y훈련)

print(f"입력 열 {len(df.columns)}개 -> 전처리 후 특징 "
      f"{전처리.transform(df훈련).shape[1]}개\n")
이름들 = 전처리.get_feature_names_out()
print("만들어진 특징 이름")
for 이름 in 이름들:
    print(f"   {이름}")

print(f"\n훈련 정확도   {전체관.score(df훈련, y훈련):.1%}")
print(f"테스트 정확도 {전체관.score(df테스트, y테스트):.1%}")
print("\n=> ⭐ `DataFrame` 을 그대로 넣었습니다. 열 이름으로 라우팅되니까요.")
print("   8단원에서 손으로 `np.c_[...]` 하던 걸 안 해도 됩니다.")

입력 열 5개 -> 전처리 후 특징 10개

만들어진 특징 이름
   숫자__나이
   숫자__소득
   숫자__사용량
   범주__도시_광주
   범주__도시_대구
   범주__도시_부산
   범주__도시_서울
   범주__요금제_베이직
   범주__요금제_엔터프라이즈
   범주__요금제_프로

훈련 정확도   78.2%
테스트 정확도 78.3%

=> ⭐ `DataFrame` 을 그대로 넣었습니다. 열 이름으로 라우팅되니까요.
   8단원에서 손으로 `np.c_[...]` 하던 걸 안 해도 됩니다.


In [9]:
# ✨ 추가 — ⚠️ 0절 장애 ④: 프로덕션에 처음 보는 범주가 나타나면
df새, y새 = 고객데이터(n=300, 시드=99, 새범주=True)      # '세종' 이 등장합니다
처음본것 = set(df새["도시"]) - set(df훈련["도시"])
print(f"프로덕션 데이터에 새로 등장한 도시: {처음본것}")
print(f"해당 행 수: {int(df새['도시'].isin(처음본것).sum())}개 / {len(df새)}개\n")

# ✅ handle_unknown="ignore" 인 경우
try:
    점수 = 전체관.score(df새, y새)
    print(f"✅ handle_unknown='ignore' : 정상 동작, 정확도 {점수:.1%}")
    변환됨 = 전처리.transform(df새[df새["도시"].isin(처음본것)])
    도시칸 = [i for i, n in enumerate(이름들) if n.startswith("범주__도시")]
    print(f"   새 범주 행의 도시 원핫 값: {변환됨[0, 도시칸]}  <- **전부 0 벡터**")
except Exception as e:
    print(f"✅ 쪽에서 예외: {type(e).__name__}")

# ❌ handle_unknown="error" (기본값) 인 경우
범주관터짐 = Pipeline([("채우기", SimpleImputer(strategy="most_frequent")),
                      ("원핫", OneHotEncoder(handle_unknown="error"))])
전처리터짐 = ColumnTransformer([("숫자", 숫자관, 숫자열), ("범주", 범주관터짐, 범주열)])
관터짐 = Pipeline([("전처리", 전처리터짐),
                  ("모델", LogisticRegression(max_iter=3000))]).fit(df훈련, y훈련)
print()
try:
    관터짐.score(df새, y새)
    print("❌ handle_unknown='error': 어째서인지 통과했습니다")
except Exception as e:
    print(f"❌ handle_unknown='error' : **{type(e).__name__}** 로 터집니다")
    print(f"   {str(e).split(chr(10))[0][:110]}")

print("\n=> ⚠️ 이게 📗 원문이 'critical for production' 이라고 한 이유입니다.")
print("   서울·부산·대구·광주로 학습한 모델에 '세종'이 들어오면 **서비스가 멈춥니다.**")
print("   `handle_unknown='ignore'` 는 0 벡터를 내놓고 **계속 동작합니다.**")
print("\n   💡 다만 0 벡터는 '아무 도시도 아님'이라는 뜻이라 예측 품질은 떨어집니다.")
print("      그래서 실무에서는 두 가지를 같이 합니다.")
print("      ① `handle_unknown='ignore'` 로 **죽지 않게** 하고")
print("      ② 새 범주 등장을 **모니터링**해서 재학습 시점을 잡습니다")

프로덕션 데이터에 새로 등장한 도시: {'세종'}
해당 행 수: 53개 / 300개

✅ handle_unknown='ignore' : 정상 동작, 정확도 78.3%
   새 범주 행의 도시 원핫 값: [0. 0. 0. 0.]  <- **전부 0 벡터**

❌ handle_unknown='error' : **ValueError** 로 터집니다
   Found unknown categories ['세종'] in column 0 during transform

=> ⚠️ 이게 📗 원문이 'critical for production' 이라고 한 이유입니다.
   서울·부산·대구·광주로 학습한 모델에 '세종'이 들어오면 **서비스가 멈춥니다.**
   `handle_unknown='ignore'` 는 0 벡터를 내놓고 **계속 동작합니다.**

   💡 다만 0 벡터는 '아무 도시도 아님'이라는 뜻이라 예측 품질은 떨어집니다.
      그래서 실무에서는 두 가지를 같이 합니다.
      ① `handle_unknown='ignore'` 로 **죽지 않게** 하고
      ② 새 범주 등장을 **모니터링**해서 재학습 시점을 잡습니다


---

## 6️⃣ 🔑 교차검증 + 파이프라인

> 📗 **원문** — "Build It: Step 3"
>
> 🔗 **여기서 2절의 누출이 구조적으로 불가능해집니다** — 9단원 3절의 K-겹과 합칩니다.

### 🔑 왜 중요한가

교차검증은 데이터를 **K번 다르게 나눕니다.** 그러면 전처리도 **K번 다르게 fit** 되어야 합니다.

| | 전처리를 언제 fit 하나 | 결과 |
| --- | --- | --- |
| ❌ 파이프라인 **없이** | 전체 데이터로 **한 번** | 각 겹의 검증 부분이 전처리에 포함됨 → **누출** |
| ✅ 파이프라인 **안에** | **겹마다 훈련 부분으로** | 검증 부분은 전처리도 못 봄 → **깨끗** |

> 📗 원문: "the scaler is fit separately on each fold's training data."

### 🧸 비유 — 모의고사를 K번 보기

모의고사를 5번 봅니다. 매번 **다른 문제 세트**가 시험 범위에서 빠지죠.
그런데 **공부 범위를 정할 때 5번 시험지를 전부 봤다면?** 5번 다 오염됩니다.

파이프라인은 **각 시험마다 그 시험지를 빼고 공부 범위를 정하게** 강제합니다.

In [10]:
# 📗 원문 Step 3 — 교차검증 + 파이프라인 (+ ✨ 겹마다 fit 되는 걸 추적)
호출기록.clear()
추적관2 = Pipeline([("추적", 추적변환기("스케일러")),
                   ("모델", LogisticRegression(max_iter=2000))])
점수들 = cross_val_score(추적관2, X숫자[온전], y[온전], cv=겹5)

fit횟수 = sum(1 for 줄 in 호출기록 if ".fit(" in 줄)
transform횟수 = sum(1 for 줄 in 호출기록 if ".transform(" in 줄)
print(f"5-겹 교차검증 중 호출 내역")
print(f"   스케일러.fit 호출      : {fit횟수}번")
print(f"   스케일러.transform 호출: {transform횟수}번")
print(f"\n처음 6줄:")
for 줄 in 호출기록[:6]:
    print(f"   {줄}")
print(f"\n=> ⭐ **fit 이 {fit횟수}번 불렸습니다 — 겹 개수만큼입니다.**")
print("   📗 원문의 'the scaler is fit separately on each fold's training data' 가 이것입니다.")
print("   행 수를 보세요. fit 은 훈련 부분(더 큰 수)에만 불립니다.")

5-겹 교차검증 중 호출 내역
   스케일러.fit 호출      : 5번
   스케일러.transform 호출: 10번

처음 6줄:
   스케일러.fit(행 1401개)
   스케일러.transform(행 1401개)
   스케일러.transform(행 351개)
   스케일러.fit(행 1401개)
   스케일러.transform(행 1401개)
   스케일러.transform(행 351개)

=> ⭐ **fit 이 5번 불렸습니다 — 겹 개수만큼입니다.**
   📗 원문의 'the scaler is fit separately on each fold's training data' 가 이것입니다.
   행 수를 보세요. fit 은 훈련 부분(더 큰 수)에만 불립니다.


In [11]:
# ✨ 추가 — 파이프라인 없이 교차검증하면 얼마나 부풀려지나 (2절 + 9단원 3절)
X무, y무 = 무의미데이터(n행=120, n열=800)

# ❌ 전처리를 전체로 미리 하고 교차검증
미리고름 = SelectKBest(f_classif, k=15).fit(X무, y무)
미리스케일 = StandardScaler().fit(미리고름.transform(X무))
X미리 = 미리스케일.transform(미리고름.transform(X무))
나쁜점수 = cross_val_score(LogisticRegression(max_iter=3000), X미리, y무, cv=겹5)

# ✅ 전부 파이프라인 안에
좋은관 = Pipeline([("고르기", SelectKBest(f_classif, k=15)),
                  ("스케일", StandardScaler()),
                  ("모델", LogisticRegression(max_iter=3000))])
좋은점수 = cross_val_score(좋은관, X무, y무, cv=겹5)

print(f"정보가 0인 데이터 ({X무.shape[0]}행 × {X무.shape[1]}열). 진짜 실력 = 50%\n")
print(f"{'방법':<36}{'겹별 점수':>26}{'평균':>9}")
print("-" * 74)
print(f"{'❌ 전체로 전처리 후 교차검증':<33}"
      f"{str(np.round(나쁜점수, 2)):>26}{나쁜점수.mean():>9.1%}")
print(f"{'✅ 전처리를 Pipeline 안에':<34}"
      f"{str(np.round(좋은점수, 2)):>26}{좋은점수.mean():>9.1%}")
print(f"{'(정답)':<61}{0.5:>9.1%}")
print(f"\n=> 부풀림 {나쁜점수.mean()-좋은점수.mean():+.1%}")
print("   ⚠️ 왼쪽은 **에러도 경고도 없이** 좋은 점수를 냈습니다. 그게 무서운 점입니다.")
print("\n💡 규칙은 하나입니다 — **전처리는 '모델의 일부'입니다.**")
print("   모델에 들어가는 모든 것을 Pipeline 에 넣으세요. 그러면 고민할 일이 없습니다.")

정보가 0인 데이터 (120행 × 800열). 진짜 실력 = 50%

방법                                                       겹별 점수       평균
--------------------------------------------------------------------------
❌ 전체로 전처리 후 교차검증                 [0.67 0.79 0.67 0.79 0.71]    72.5%
✅ 전처리를 Pipeline 안에                [0.54 0.42 0.38 0.58 0.58]    50.0%
(정답)                                                             50.0%

=> 부풀림 +22.5%
   ⚠️ 왼쪽은 **에러도 경고도 없이** 좋은 점수를 냈습니다. 그게 무서운 점입니다.

💡 규칙은 하나입니다 — **전처리는 '모델의 일부'입니다.**
   모델에 들어가는 모든 것을 Pipeline 에 넣으세요. 그러면 고민할 일이 없습니다.


In [12]:
# ✨ 추가 — 12단원과 합치기: 파이프라인 전체를 하이퍼파라미터 탐색 대상으로
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

탐색관 = Pipeline([("전처리", ColumnTransformer([
                      ("숫자", Pipeline([("채우기", SimpleImputer()),
                                        ("스케일", StandardScaler())]), 숫자열),
                      ("범주", Pipeline([("채우기", SimpleImputer(strategy="most_frequent")),
                                        ("원핫", OneHotEncoder(handle_unknown="ignore"))]),
                       범주열)])),
                  ("모델", GradientBoostingClassifier(random_state=0))])

# ⭐ 전처리 단계의 하이퍼파라미터까지 같이 탐색합니다 (이중 밑줄로 경로 지정)
격자 = {
    "전처리__숫자__채우기__strategy": ["mean", "median"],
    "모델__learning_rate": [0.05, 0.2],
    "모델__max_depth": [2, 3],
}
탐색 = GridSearchCV(탐색관, 격자, cv=3, scoring="accuracy", n_jobs=-1)
탐색.fit(df훈련, y훈련)

print(f"탐색한 조합 수: {len(탐색.cv_results_['params'])}\n")
print(f"최적 설정")
for k, v in 탐색.best_params_.items():
    print(f"   {k:<36} = {v}")
print(f"\n교차검증 정확도: {탐색.best_score_:.1%}")
print(f"테스트 정확도  : {탐색.score(df테스트, y테스트):.1%}")

print("\n=> ⭐ **전처리 방식('mean' vs 'median')도 하이퍼파라미터입니다.**")
print("   파이프라인에 넣으면 12단원의 탐색 도구가 **전처리까지 같이 튜닝**합니다.")
print("   그리고 겹마다 전처리가 새로 fit 되니 **누출 걱정이 없습니다.**")
print("\n   ⚠️ 파이프라인에 안 넣으면 어떻게 될까요? 전처리를 미리 정해 놓고")
print("      모델만 튜닝하게 됩니다. **전처리가 최적이 아닐 수 있는데도요.**")

탐색한 조합 수: 8

최적 설정
   모델__learning_rate                    = 0.2
   모델__max_depth                        = 2
   전처리__숫자__채우기__strategy               = mean

교차검증 정확도: 81.4%
테스트 정확도  : 80.7%

=> ⭐ **전처리 방식('mean' vs 'median')도 하이퍼파라미터입니다.**
   파이프라인에 넣으면 12단원의 탐색 도구가 **전처리까지 같이 튜닝**합니다.
   그리고 겹마다 전처리가 새로 fit 되니 **누출 걱정이 없습니다.**

   ⚠️ 파이프라인에 안 넣으면 어떻게 될까요? 전처리를 미리 정해 놓고
      모델만 튜닝하게 됩니다. **전처리가 최적이 아닐 수 있는데도요.**


---

## 7️⃣ 직렬화 — 학습/서빙 불일치 막기

> 📗 **원문** — 0절 장애 ②·③ + "Common Pipeline Mistakes"
>
> 🔗 **앞 절과 이어서** — 6절까지는 **학습할 때** 올바르게 하는 법이었습니다.
> 이제 **배포하고 나서도** 올바르게 유지하는 법입니다.

### 📗 0절 장애 ②·③을 다시 보세요

| 장애 | 내용 |
| --- | --- |
| ② | **스케일링 파라미터를 저장하지 않아** 추론 때 다른 통계를 씁니다 |
| ③ | 특징 공학 코드를 **학습용·서빙용으로 복사했는데 사본이 갈라졌습니다** |

### 🧸 비유 — 열쇠와 자물쇠를 따로 보관하기

자물쇠(모델)만 배포하고 **열쇠(전처리 통계)를 안 보냈다면?** 열 수 없습니다.
그래서 서빙 쪽에서 **열쇠를 새로 깎습니다.** 모양이 조금 다릅니다. **조용히 틀린 예측이 나옵니다.**

**파이프라인은 자물쇠와 열쇠를 한 상자에 담습니다.** 하나의 파일로 저장되죠.

### 📗 해법 — 파이프라인 전체를 하나로 저장

```python
import joblib
joblib.dump(pipeline, "model.joblib")        # 전처리 + 모델이 한 덩어리
loaded = joblib.load("model.joblib")
loaded.predict(new_data)                      # 같은 전처리가 자동 적용
```

> 📗 원문의 "Common Pipeline Mistakes" 표:
> **"Training/serving skew — One Pipeline object for both"**

In [13]:
# ✨ 추가 — 원문 연습문제 3번: 직렬화하고 다시 불러 예측이 같은지 확인
import joblib

경로 = os.path.join(작업방, "churn-pipeline.joblib")
joblib.dump(전체관, 경로)
크기KB = os.path.getsize(경로) / 1024
불러온것 = joblib.load(경로)

원래예측 = 전체관.predict(df테스트)
원래확률 = 전체관.predict_proba(df테스트)[:, 1]
불러온예측 = 불러온것.predict(df테스트)
불러온확률 = 불러온것.predict_proba(df테스트)[:, 1]

print(f"저장: {경로}")
print(f"파일 크기: {크기KB:.1f} KB\n")
print(f"라벨 예측이 완전히 같은가 : {np.array_equal(원래예측, 불러온예측)}")
print(f"확률 예측의 최대 차이     : {np.abs(원래확률 - 불러온확률).max():.2e}")
print(f"\n=> ⭐ **전처리 통계까지 전부 같이 저장됐습니다.**")

# 안에 무엇이 들었는지 확인합니다
스케일러 = 불러온것.named_steps["전처리"].named_transformers_["숫자"].named_steps["스케일"]
채우기 = 불러온것.named_steps["전처리"].named_transformers_["숫자"].named_steps["채우기"]
원핫 = 불러온것.named_steps["전처리"].named_transformers_["범주"].named_steps["원핫"]
print(f"\n저장된 파일 안에 들어 있는 '학습된 통계'")
print(f"   결측 채울 값 (중앙값) : {np.round(채우기.statistics_, 2)}")
print(f"   스케일러 평균        : {np.round(스케일러.mean_, 2)}")
print(f"   스케일러 표준편차     : {np.round(스케일러.scale_, 2)}")
print(f"   원핫이 아는 범주      : {[list(c) for c in 원핫.categories_]}")
print("\n=> 이 값들이 **파일 안에** 있습니다. 서빙 쪽에서 다시 계산할 필요가 없죠.")
print("   📗 원문 장애 ②(파라미터 미저장)와 ③(사본 분기)이 **둘 다 해결됩니다.**")

저장: /var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2/churn-pipeline.joblib
파일 크기: 4.7 KB

라벨 예측이 완전히 같은가 : True
확률 예측의 최대 차이     : 0.00e+00

=> ⭐ **전처리 통계까지 전부 같이 저장됐습니다.**

저장된 파일 안에 들어 있는 '학습된 통계'
   결측 채울 값 (중앙값) : [   46.   26594.34    50.06]
   스케일러 평균        : [   46.14 40263.2     61.05]
   스케일러 표준편차     : [   15.15 42749.42    42.66]
   원핫이 아는 범주      : [['광주', '대구', '부산', '서울'], ['베이직', '엔터프라이즈', '프로']]

=> 이 값들이 **파일 안에** 있습니다. 서빙 쪽에서 다시 계산할 필요가 없죠.
   📗 원문 장애 ②(파라미터 미저장)와 ③(사본 분기)이 **둘 다 해결됩니다.**


In [14]:
# ✨ 추가 — ⚠️ 전처리를 파이프라인 밖에 두면 어떻게 깨지나 (장애 ③ 재현)
print("상황: 학습 스크립트와 서빙 스크립트가 전처리 코드를 '각자' 가지고 있습니다\n")

# 학습 쪽: 중앙값으로 채우고 표준화
학습쪽채움 = float(np.nanmedian(df훈련["나이"]))
학습쪽평균 = float(df훈련["소득"].mean())
학습쪽표준 = float(df훈련["소득"].std())

# 서빙 쪽: 같은 코드를 '복사'했는데 한 줄이 갈라졌습니다 (median -> mean)
서빙쪽채움 = float(np.nanmean(df테스트["나이"]))      # ⚠️ median 이어야 하는데 mean
서빙쪽평균 = float(df테스트["소득"].mean())           # ⚠️ 테스트 통계로 계산
서빙쪽표준 = float(df테스트["소득"].std())

print(f"{'전처리 통계':<20}{'학습 쪽':>14}{'서빙 쪽':>14}{'차이':>12}")
print("-" * 62)
for 이름, a, b in [("나이 채울 값", 학습쪽채움, 서빙쪽채움),
                  ("소득 평균", 학습쪽평균, 서빙쪽평균),
                  ("소득 표준편차", 학습쪽표준, 서빙쪽표준)]:
    print(f"{이름:<19}{a:>14.2f}{b:>14.2f}{abs(a-b)/max(abs(a),1e-9):>11.1%}")

# 두 방식의 예측이 얼마나 다른지
def 손전처리(d, 채움, 평균, 표준):
    나이 = d["나이"].fillna(채움).to_numpy()
    소득 = ((d["소득"] - 평균) / 표준).to_numpy()
    사용량 = d["사용량"].to_numpy()
    return np.c_[나이, 소득, 사용량]

모델손 = LogisticRegression(max_iter=3000).fit(
    손전처리(df훈련, 학습쪽채움, 학습쪽평균, 학습쪽표준), y훈련)
일치하는경우 = 모델손.predict(손전처리(df테스트, 학습쪽채움, 학습쪽평균, 학습쪽표준))
갈라진경우 = 모델손.predict(손전처리(df테스트, 서빙쪽채움, 서빙쪽평균, 서빙쪽표준))

print(f"\n{'서빙 방식':<28}{'테스트 정확도':>14}")
print("-" * 44)
print(f"{'학습과 같은 통계 사용':<25}{np.mean(일치하는경우 == y테스트):>14.1%}")
print(f"{'⚠️ 사본이 갈라진 경우':<25}{np.mean(갈라진경우 == y테스트):>14.1%}")
print(f"\n예측이 달라진 행: {int((일치하는경우 != 갈라진경우).sum())}개 / {len(df테스트)}개 "
      f"({np.mean(일치하는경우 != 갈라진경우):.1%})")
print("\n=> ⚠️ **에러가 안 납니다.** 그냥 예측이 조용히 달라집니다.")
print("   이게 📗 원문의 'training/serving skew' 입니다. 찾기 가장 어려운 버그죠.")
print("\n   💡 파이프라인을 쓰면 **이 상황이 생길 수 없습니다.**")
print("      전처리 코드가 두 벌 존재하지 않으니까요. 한 파일에서 불러 쓰면 끝입니다.")

상황: 학습 스크립트와 서빙 스크립트가 전처리 코드를 '각자' 가지고 있습니다

전처리 통계                        학습 쪽          서빙 쪽          차이
--------------------------------------------------------------
나이 채울 값                     46.00         46.40       0.9%
소득 평균                    40263.20      40827.74       1.4%
소득 표준편차                  42764.70      44941.24       5.1%

서빙 방식                              테스트 정확도
--------------------------------------------
학습과 같은 통계 사용                      64.5%
⚠️ 사본이 갈라진 경우                     64.3%

예측이 달라진 행: 3개 / 600개 (0.5%)

=> ⚠️ **에러가 안 납니다.** 그냥 예측이 조용히 달라집니다.
   이게 📗 원문의 'training/serving skew' 입니다. 찾기 가장 어려운 버그죠.

   💡 파이프라인을 쓰면 **이 상황이 생길 수 없습니다.**
      전처리 코드가 두 벌 존재하지 않으니까요. 한 파일에서 불러 쓰면 끝입니다.


---

## 8️⃣ 실험 추적 (MLflow)

> 📗 **원문** — "Experiment Tracking" + "Model Versioning"
>
> 🔗 **앞 절과 이어서** — 7절에서 파이프라인을 **저장**했습니다.
> 그런데 12단원처럼 **수십~수백 번 실험**하면 파일이 수백 개가 됩니다.
> **어느 파일이 어떤 설정이었는지** 어떻게 기억하죠?

> 📗 원문: "A pipeline makes training reproducible, but you also need to track
> what happened across experiments: which hyperparameters were used,
> which dataset version, what the metrics were, which code was running."

### 📗 MLflow 기본 형태

```python
import mlflow

with mlflow.start_run():
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("n_estimators", 100)

    pipe.fit(X_train, y_train)
    accuracy = pipe.score(X_test, y_test)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(pipe, "model")
```

### 📗 기록되는 네 가지

| 종류 | 예시 |
| --- | --- |
| **파라미터** | `max_depth=5`, `learning_rate=0.1` |
| **지표** | `accuracy=0.87`, `auc=0.93` |
| **아티팩트** | 모델 파일, 그림, 설정 파일 |
| **모델** | 파이프라인 전체 (7절의 그 객체) |

### 📗 모델 레지스트리 — 버전 관리

| 기능 | 하는 일 |
| --- | --- |
| **버전 추적** | 저장할 때마다 버전 번호가 붙습니다 |
| **단계 전환** | "Staging" → "Production" → "Archived" |
| **승인 절차** | 프로덕션으로는 **명시적으로 승격**해야 합니다 |
| **롤백** | 이전 버전으로 **즉시** 되돌리기 |

> ⚠️ **데이터 버전 관리는 git으로 안 됩니다.** 큰 파일을 못 다루니까요.
> **DVC** 가 그걸 합니다 — 데이터는 S3 등에 두고, **해시만 git에** 넣습니다.
> 그러면 **커밋 하나가 코드와 데이터를 둘 다 고정**합니다.

In [15]:
# 📗 원문 — MLflow 로 실험을 추적합니다 (원문 연습문제 5번)
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import mlflow
import mlflow.sklearn

# ⚠️ MLflow 3.x 부터 파일 백엔드(./mlruns)가 폐기 예정입니다.
#    SQLite 를 쓰라고 공식 권고합니다. 로컬 실험에는 이게 가장 간단합니다
추적DB = os.path.join(작업방, "mlflow.db")
아티팩트 = os.path.join(작업방, "artifacts")
os.makedirs(아티팩트, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{추적DB}")
mlflow.set_experiment("고객이탈-파이프라인")

설정들 = [
    {"모델__learning_rate": 0.05, "모델__max_depth": 2},
    {"모델__learning_rate": 0.1, "모델__max_depth": 3},
    {"모델__learning_rate": 0.2, "모델__max_depth": 3},
    {"모델__learning_rate": 0.3, "모델__max_depth": 5},
    {"모델__learning_rate": 0.5, "모델__max_depth": 8},
]

실행기록 = []
for i, 설정 in enumerate(설정들, 1):
    with mlflow.start_run(run_name=f"실행-{i}"):
        관 = Pipeline([("전처리", ColumnTransformer([
                          ("숫자", 숫자관, 숫자열), ("범주", 범주관, 범주열)])),
                      ("모델", GradientBoostingClassifier(n_estimators=120, random_state=0))])
        관.set_params(**설정)
        관.fit(df훈련, y훈련)

        훈련점수 = 관.score(df훈련, y훈련)
        테스트점수 = 관.score(df테스트, y테스트)

        # ① 파라미터 기록
        for k, v in 설정.items():
            mlflow.log_param(k.split("__")[-1], v)
        mlflow.log_param("데이터_행수", len(df훈련))
        # ② 지표 기록
        mlflow.log_metric("훈련_정확도", 훈련점수)
        mlflow.log_metric("테스트_정확도", 테스트점수)
        mlflow.log_metric("과적합_간격", 훈련점수 - 테스트점수)
        # ③ 모델(파이프라인 전체) 기록
        # ⚠️ MLflow 3.x 는 skops 로 직렬화하면서 '신뢰할 타입'을 명시하라고 요구합니다.
        #    모델 파일 역직렬화는 실제 공격 경로입니다 (아래 설명 참고)
        mlflow.sklearn.log_model(관, name="model",
                                 skops_trusted_types=["sklearn.tree._tree.Tree",
                                                      "numpy.dtype",
                                                      "sklearn.ensemble._gb_losses."
                                                      "BinomialDeviance"])
        실행기록.append((i, 설정, 훈련점수, 테스트점수))

print(f"MLflow 실행 {len(실행기록)}건 기록 완료\n")
print(f"{'실행':>5}{'학습률':>9}{'최대깊이':>10}{'훈련':>9}{'테스트':>9}{'간격':>9}")
print("-" * 54)
for i, 설정, tr, te in 실행기록:
    print(f"{i:>5}{설정['모델__learning_rate']:>9}{설정['모델__max_depth']:>10}"
          f"{tr:>9.1%}{te:>9.1%}{tr-te:>9.1%}")

최고 = max(실행기록, key=lambda r: r[3])
print(f"\n최고 테스트 정확도: 실행 {최고[0]} ({최고[3]:.1%})")
print(f"   설정: { {k.split('__')[-1]: v for k, v in 최고[1].items()} }")

print("\n⚠️ 위 코드에 `skops_trusted_types=[...]` 가 있는 이유")
print("   MLflow 3.x 는 모델을 `skops` 로 직렬화하는데, **신뢰할 타입을 명시**하게 합니다.")
print("   왜냐면 `sklearn.tree._tree.Tree` 는 노드 인덱스를 **경계 검사 없이** 씁니다.")
print("   악의적으로 만든 모델 파일을 불러 `.predict()` 하면 프로세스가 죽거나")
print("   메모리를 범위 밖으로 읽을 수 있습니다.")
print("\n   💡 7절에서 `joblib.dump/load` 를 썼죠. **joblib(=pickle)은 더 위험합니다.**")
print("      pickle 역직렬화는 **임의 코드 실행**이 가능합니다.")
print("      => **내가 만들지 않은 모델 파일은 절대 불러오지 마세요.**")
print("      프로덕션에서는 모델 레지스트리에 체크섬을 두고 검증하는 게 표준입니다.")

2026/10/04 22:09:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/04 22:09:30 INFO mlflow.store.db.utils: Updating database tables
2026/10/04 22:09:30 INFO mlflow.tracking.fluent: Experiment with name '고객이탈-파이프라인' does not exist. Creating a new experiment.


MLflow 실행 5건 기록 완료

   실행      학습률      최대깊이       훈련      테스트       간격
------------------------------------------------------
    1     0.05         2    83.7%    81.7%     2.0%
    2      0.1         3    89.4%    81.8%     7.5%
    3      0.2         3    93.4%    80.2%    13.2%
    4      0.3         5   100.0%    78.3%    21.7%
    5      0.5         8   100.0%    76.3%    23.7%

최고 테스트 정확도: 실행 2 (81.8%)
   설정: {'learning_rate': 0.1, 'max_depth': 3}

⚠️ 위 코드에 `skops_trusted_types=[...]` 가 있는 이유
   MLflow 3.x 는 모델을 `skops` 로 직렬화하는데, **신뢰할 타입을 명시**하게 합니다.
   왜냐면 `sklearn.tree._tree.Tree` 는 노드 인덱스를 **경계 검사 없이** 씁니다.
   악의적으로 만든 모델 파일을 불러 `.predict()` 하면 프로세스가 죽거나
   메모리를 범위 밖으로 읽을 수 있습니다.

   💡 7절에서 `joblib.dump/load` 를 썼죠. **joblib(=pickle)은 더 위험합니다.**
      pickle 역직렬화는 **임의 코드 실행**이 가능합니다.
      => **내가 만들지 않은 모델 파일은 절대 불러오지 마세요.**
      프로덕션에서는 모델 레지스트리에 체크섬을 두고 검증하는 게 표준입니다.


In [16]:
# ✨ 추가 — 기록을 다시 읽어 비교합니다 (MLflow UI 가 하는 일을 코드로)
표 = mlflow.search_runs(experiment_names=["고객이탈-파이프라인"])
보여줄열 = [c for c in 표.columns
            if c.startswith("metrics.") or c.startswith("params.")]
정리표 = 표[["run_id"] + 보여줄열].copy()
정리표["run_id"] = 정리표["run_id"].str[:8]
정리표 = 정리표.sort_values("metrics.테스트_정확도", ascending=False)

print("mlflow.search_runs() 로 읽은 기록 (테스트 정확도 내림차순)\n")
print(정리표.to_string(index=False))

print(f"\n=> ⭐ **실험을 다 끝낸 뒤에도 무엇을 어떻게 돌렸는지 알 수 있습니다.**")
print(f"   run_id 로 그 시점의 **모델 파일까지** 꺼낼 수 있습니다.")

# 최고 실행의 모델을 꺼내 예측해 봅니다
최고run = 표.sort_values("metrics.테스트_정확도", ascending=False).iloc[0]
복원모델 = mlflow.sklearn.load_model(f"runs:/{최고run['run_id']}/model")
print(f"\n최고 실행({최고run['run_id'][:8]})의 모델을 꺼내 예측:")
print(f"   테스트 정확도 {복원모델.score(df테스트, y테스트):.1%}")
print(f"   기록된 값     {최고run['metrics.테스트_정확도']:.1%}")
print(f"   => 일치하면 **완전히 재현된 것**입니다.")

print(f"\n💡 실무에서는 터미널에서 아래 명령으로 브라우저 대시보드를 띄워 비교합니다.")
print(f"   mlflow ui --backend-store-uri sqlite:///{추적DB}")
print(f"   위 표를 그림으로 보는 셈이죠.")
print(f"\n   ⚠️ MLflow 3.x 부터 `./mlruns` 파일 백엔드가 **폐기 예정**입니다.")
print(f"      `sqlite:///...` 로 쓰세요. 기존 데이터는 `mlflow migrate-filestore` 로 옮깁니다.")
print("\n   ⚠️ 그리고 📗 원문이 짚는 것: **코드 버전도 기록해야 합니다.**")
print("      MLflow 는 git 커밋 해시를 자동으로 남깁니다 (git 저장소 안에서 실행하면).")
print("      '파라미터는 같은데 결과가 다르다' 면 코드가 바뀐 겁니다.")

mlflow.search_runs() 로 읽은 기록 (테스트 정확도 내림차순)

  run_id  metrics.테스트_정확도  metrics.훈련_정확도  metrics.과적합_간격 params.learning_rate params.데이터_행수 params.max_depth
b213524e         0.818333        0.893571        0.075238                  0.1          1400                3
4da0feb0         0.816667        0.837143        0.020476                 0.05          1400                2
e45fba27         0.801667        0.933571        0.131905                  0.2          1400                3
ec6b8aa9         0.783333        1.000000        0.216667                  0.3          1400                5
6bf682d1         0.763333        1.000000        0.236667                  0.5          1400                8

=> ⭐ **실험을 다 끝낸 뒤에도 무엇을 어떻게 돌렸는지 알 수 있습니다.**
   run_id 로 그 시점의 **모델 파일까지** 꺼낼 수 있습니다.



최고 실행(b213524e)의 모델을 꺼내 예측:
   테스트 정확도 81.8%
   기록된 값     81.8%
   => 일치하면 **완전히 재현된 것**입니다.

💡 실무에서는 터미널에서 아래 명령으로 브라우저 대시보드를 띄워 비교합니다.
   mlflow ui --backend-store-uri sqlite:////var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2/mlflow.db
   위 표를 그림으로 보는 셈이죠.

   ⚠️ MLflow 3.x 부터 `./mlruns` 파일 백엔드가 **폐기 예정**입니다.
      `sqlite:///...` 로 쓰세요. 기존 데이터는 `mlflow migrate-filestore` 로 옮깁니다.

   ⚠️ 그리고 📗 원문이 짚는 것: **코드 버전도 기록해야 합니다.**
      MLflow 는 git 커밋 해시를 자동으로 남깁니다 (git 저장소 안에서 실행하면).
      '파라미터는 같은데 결과가 다르다' 면 코드가 바뀐 겁니다.


---

## 9️⃣ 재현성 — 네 가지 조건

> 📗 **원문** — "Reproducible Experiments" + "Data Versioning with DVC"
>
> 🔗 **앞 절과 이어서** — 8절에서 **기록**을 했습니다.
> 그런데 기록이 있어도 **같은 결과가 안 나오면** 의미가 없습니다.

### 📗 재현 가능한 실험의 네 조건

| # | 조건 | 도구 |
| --- | --- | --- |
| 1 | **난수 시드 고정** | `random`, `numpy`, 프레임워크(torch 등) 전부 |
| 2 | **의존성 고정** | `requirements.txt` 또는 `poetry.lock` 에 **정확한 버전** |
| 3 | **데이터 버전 관리** | **DVC** 또는 유사 도구 |
| 4 | **설정 파일** | 하이퍼파라미터를 **코드에 박지 말고** config로 |

### 🧸 비유 — 요리를 똑같이 재현하기

0절에서 "레시피를 글로 적자"고 했습니다. 그런데 적어도 안 되는 게 있습니다.

| 적어야 할 것 | ML에서는 |
| --- | --- |
| 재료의 **정확한 양** | 설정 파일 (조건 4) |
| **어느 농장 재료**인지 | 데이터 버전 (조건 3) |
| **어떤 오븐** 모델인지 | 의존성 버전 (조건 2) |
| **주사위를 어떻게 굴렸나** | 난수 시드 (조건 1) |

> ⚠️ **조건 1을 빼먹기 가장 쉽습니다.** `random_state` 를 안 넣은 모델 하나 때문에
> 매번 결과가 달라지는 일이 흔합니다. 아래에서 확인합니다.

In [17]:
# 📗 원문 — 시드 고정 함수 (조건 1)
def 시드고정(시드=42):
    random.seed(시드)
    np.random.seed(시드)
    try:
        import torch
        torch.manual_seed(시드)
        torch.cuda.manual_seed_all(시드)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass
    return 시드

# ✨ 추가 — random_state 를 빼먹으면 어떻게 되나
from sklearn.ensemble import RandomForestClassifier

print("같은 코드를 3번 돌려 봅니다\n")
print(f"{'설정':<34}{'1회':>9}{'2회':>9}{'3회':>9}{'전부 같나':>11}")
print("-" * 74)

for 이름, 만들기, 시드고정하나 in [
        ("❌ random_state 없음", lambda: RandomForestClassifier(n_estimators=50), False),
        ("⚠️ np.random.seed 만 고정", lambda: RandomForestClassifier(n_estimators=50), True),
        ("✅ random_state=0", lambda: RandomForestClassifier(n_estimators=50,
                                                            random_state=0), False)]:
    점수들 = []
    for _ in range(3):
        if 시드고정하나:
            시드고정(42)
        관 = Pipeline([("전처리", ColumnTransformer([("숫자", 숫자관, 숫자열),
                                                   ("범주", 범주관, 범주열)])),
                      ("모델", 만들기())]).fit(df훈련, y훈련)
        점수들.append(round(관.score(df테스트, y테스트), 6))
    같나 = "✔ 예" if len(set(점수들)) == 1 else "✘ 아니오"
    print(f"{이름:<33}{점수들[0]:>9.4f}{점수들[1]:>9.4f}{점수들[2]:>9.4f}{같나:>11}")

print("\n=> ⚠️ **`random_state` 를 안 넣으면 매번 다른 결과가 나옵니다.**")
print("   그리고 `np.random.seed()` 만 고정해도 **부족합니다.**")
print("   sklearn 추정기는 자기 `random_state` 를 따로 쓰기 때문입니다.")
print("\n   💡 규칙: **난수를 쓰는 모든 객체에 `random_state` 를 명시하세요.**")
print("      train_test_split, KFold, 모델, 샘플러 전부입니다.")

같은 코드를 3번 돌려 봅니다

설정                                       1회       2회       3회      전부 같나
--------------------------------------------------------------------------
❌ random_state 없음                   0.7933   0.7800   0.7800      ✘ 아니오
⚠️ np.random.seed 만 고정              0.7683   0.7683   0.7683        ✔ 예
✅ random_state=0                    0.7717   0.7717   0.7717        ✔ 예

=> ⚠️ **`random_state` 를 안 넣으면 매번 다른 결과가 나옵니다.**
   그리고 `np.random.seed()` 만 고정해도 **부족합니다.**
   sklearn 추정기는 자기 `random_state` 를 따로 쓰기 때문입니다.

   💡 규칙: **난수를 쓰는 모든 객체에 `random_state` 를 명시하세요.**
      train_test_split, KFold, 모델, 샘플러 전부입니다.


In [18]:
# ✨ 추가 — 조건 2·3·4를 실제 파일로 만들어 봅니다
import sklearn, platform

# 조건 4 — 설정 파일 (하이퍼파라미터를 코드에서 분리)
설정 = {
    "데이터": {"행수": len(df), "숫자열": 숫자열, "범주열": 범주열},
    "전처리": {"숫자_채우기": "median", "범주_채우기": "most_frequent",
               "원핫_미지범주": "ignore"},
    "모델": {"종류": "GradientBoostingClassifier", "n_estimators": 120,
             "learning_rate": 0.1, "max_depth": 3, "random_state": 0},
    "분할": {"test_size": 0.3, "random_state": 0, "stratify": True},
    "시드": 42,
}
설정경로 = os.path.join(작업방, "config.json")
with open(설정경로, "w", encoding="utf-8") as fh:
    _json.dump(설정, fh, ensure_ascii=False, indent=2)

# 조건 2 — 의존성 버전 고정
의존성 = {"python": platform.python_version(), "numpy": np.__version__,
          "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
          "mlflow": mlflow.__version__, "joblib": joblib.__version__}
의존성경로 = os.path.join(작업방, "requirements-lock.txt")
with open(의존성경로, "w", encoding="utf-8") as fh:
    for k, v in 의존성.items():
        fh.write(f"{k}=={v}\n")

# 조건 3 — 데이터 해시 (DVC 가 하는 일의 핵심)
import hashlib
데이터해시 = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:16]

print("재현성 네 조건을 파일로 남겼습니다\n")
print(f"조건 1 — 난수 시드: {설정['시드']} (코드에서 시드고정() 호출)")
print(f"조건 2 — 의존성 고정: {의존성경로}")
for k, v in 의존성.items():
    print(f"     {k}=={v}")
print(f"\n조건 3 — 데이터 해시: {데이터해시}")
print("     DVC 는 이 해시를 .dvc 파일로 git 에 넣고, 실제 데이터는 S3 등에 둡니다")
print("     => **커밋 하나가 코드와 데이터를 둘 다 고정합니다**")
print(f"\n조건 4 — 설정 파일: {설정경로}")
print(_json.dumps(설정, ensure_ascii=False, indent=2)[:420] + " ...")

print("\n=> ⭐ 이 네 개가 있으면 **몇 달 뒤 누구든 같은 결과를 냅니다.**")
print("   0절의 '한 달 뒤 다른 결과가 나왔다' 가 바로 이것들이 없어서 생긴 일입니다.")
print("\n   💡 그리고 8절의 MLflow 와 합치면 완성됩니다 —")
print("      MLflow 가 run 마다 이 네 개를 전부 아티팩트로 붙여 두면 되죠.")

재현성 네 조건을 파일로 남겼습니다

조건 1 — 난수 시드: 42 (코드에서 시드고정() 호출)
조건 2 — 의존성 고정: /var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2/requirements-lock.txt
     python==3.11.8
     numpy==1.26.4
     pandas==2.1.4
     scikit-learn==1.2.2
     mlflow==3.16.1
     joblib==1.2.0

조건 3 — 데이터 해시: 115cd34ed6688475
     DVC 는 이 해시를 .dvc 파일로 git 에 넣고, 실제 데이터는 S3 등에 둡니다
     => **커밋 하나가 코드와 데이터를 둘 다 고정합니다**

조건 4 — 설정 파일: /var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2/config.json
{
  "데이터": {
    "행수": 2000,
    "숫자열": [
      "나이",
      "소득",
      "사용량"
    ],
    "범주열": [
      "도시",
      "요금제"
    ]
  },
  "전처리": {
    "숫자_채우기": "median",
    "범주_채우기": "most_frequent",
    "원핫_미지범주": "ignore"
  },
  "모델": {
    "종류": "GradientBoostingClassifier",
    "n_estimators": 120,
    "learning_rate": 0.1,
    "max_depth": 3,
    "random_state": 0
  },
  "분할": {
    "test_size": 0.3,
    "random_ ...

=> ⭐ 이 네 개가 있으면 **몇 달 뒤 누구든 같은 결과를 냅니다.**
   0절의 '한 달 뒤 다른 결과가 나왔다' 

---

## 🔟 노트북에서 프로덕션으로

> 📗 **원문** — "From Notebook to Production Pipeline"
>
> 🔗 **마무리** — 0~9절의 도구를 **실제 작업 순서**로 늘어놓습니다.

### 📗 8단계 경로

| # | 단계 | 하는 일 | 이 노트북 |
| --- | --- | --- | --- |
| 1 | **노트북 탐색** | 빠른 실험, 시각화, 특징 아이디어 | 8단원 |
| 2 | **함수로 추출** | 전처리·특징 공학·평가를 모듈로 | 3절 |
| 3 | **Pipeline 조립** | 변환을 sklearn Pipeline 으로 연결 | 4·5절 |
| 4 | **설정 관리** | 하이퍼파라미터를 YAML/JSON 으로 | 9절 조건 4 |
| 5 | **실험 추적** | MLflow 또는 wandb 로깅 추가 | 8절 |
| 6 | **데이터 검증** | 학습 전 스키마·분포·결측 패턴 확인 | **아래에서 만듭니다** |
| 7 | **테스트** | 변환기 단위 테스트, 파이프라인 통합 테스트 | **아래에서 만듭니다** |
| 8 | **배포** | 파이프라인 직렬화 → API 래핑 → 컨테이너화 | 7절 |

### ⚠️ 📗 흔한 파이프라인 실수 (원문 표)

| 실수 | 왜 나쁜가 | 해법 | 이 노트북 |
| --- | --- | --- | --- |
| **나누기 전에 전체로 fit** | 데이터 누출 | Pipeline + `cross_val_score` | 2·6절 |
| **특징 공학을 파이프라인 밖에** | 학습·서빙 변환이 달라짐 | 모든 변환을 Pipeline 안에 | 7절 |
| **미지 범주 미처리** | 프로덕션에서 크래시 | `OneHotEncoder(handle_unknown="ignore")` | 5절 |
| **열 이름 하드코딩** | 스키마 변경 시 깨짐 | config의 열 목록 사용 | 9절 조건 4 |
| **데이터 검증 없음** | 나쁜 데이터에 조용히 틀린 예측 | **예측 전 스키마 검사** | ⬇️ 6단계 |
| **학습/서빙 불일치** | 프로덕션에서 다른 특징을 봄 | **둘 다 하나의 Pipeline** | 7절 |

위 표에서 **아직 안 해 본 것이 '데이터 검증'과 '테스트'** 입니다. 지금 만듭니다.

In [19]:
# ✨ 추가 — 6단계: 데이터 검증 (예측 전 스키마 검사)
class 데이터검증기:
    # 학습 시점의 스키마를 기억해 두고, 예측 전에 비교합니다
    def __init__(self, 숫자열, 범주열):
        self.숫자열, self.범주열 = 숫자열, 범주열

    def 학습(self, df):
        self.열순서 = list(df.columns)
        self.자료형 = {c: str(df[c].dtype) for c in df.columns}
        self.범위 = {c: (float(df[c].min()), float(df[c].max())) for c in self.숫자열}
        self.범주값 = {c: set(df[c].dropna().unique()) for c in self.범주열}
        self.결측률 = {c: float(df[c].isna().mean()) for c in df.columns}
        return self

    def 검사(self, df):
        문제 = []
        빠진열 = set(self.열순서) - set(df.columns)
        if 빠진열:
            문제.append(("치명", f"열이 없습니다: {sorted(빠진열)}"))
        새열 = set(df.columns) - set(self.열순서)
        if 새열:
            문제.append(("경고", f"처음 보는 열: {sorted(새열)}"))

        for c in self.숫자열:
            if c not in df.columns:
                continue
            lo, hi = self.범위[c]
            폭 = hi - lo
            밖 = int(((df[c] < lo - 0.5 * 폭) | (df[c] > hi + 0.5 * 폭)).sum())
            if 밖:
                문제.append(("경고", f"'{c}' 가 학습 범위를 크게 벗어난 행 {밖}개"))

        for c in self.범주열:
            if c not in df.columns:
                continue
            새값 = set(df[c].dropna().unique()) - self.범주값[c]
            if 새값:
                문제.append(("경고", f"'{c}' 에 처음 보는 값: {sorted(새값)}"))

        for c in df.columns:
            if c in self.결측률:
                현재 = float(df[c].isna().mean())
                if 현재 > max(self.결측률[c] * 3, self.결측률[c] + 0.1):
                    문제.append(("경고", f"'{c}' 결측률 급증: "
                                        f"{self.결측률[c]:.1%} -> {현재:.1%}"))
        return 문제


검증기 = 데이터검증기(숫자열, 범주열).학습(df훈련)

print("[검사 1] 정상 데이터")
문제 = 검증기.검사(df테스트)
print(f"   발견된 문제: {len(문제)}개" if 문제 else "   ✔ 문제 없음")

print("\n[검사 2] 프로덕션에서 흔한 이상들을 섞은 데이터")
망친df = df새.copy()
망친df.loc[:30, "나이"] = np.nan                   # 결측률 급증
망친df.loc[0, "소득"] = 5_000_000.0                # 범위 밖 값
망친df["신규열"] = 1.0                              # 처음 보는 열
망친df = 망친df.drop(columns=["사용량"])             # 열 누락
for 등급, 메시지 in 검증기.검사(망친df):
    표시 = "🔴 치명" if 등급 == "치명" else "🟡 경고"
    print(f"   {표시}  {메시지}")

print("\n=> ⭐ **예측을 하기 전에 이걸 잡아야 합니다.**")
print("   📗 원문: 'No data validation -> Silently wrong predictions on bad data'")
print("   '치명'이면 예측을 거부하고, '경고'면 기록하고 계속 — 이게 보통의 정책입니다.")

[검사 1] 정상 데이터
   ✔ 문제 없음

[검사 2] 프로덕션에서 흔한 이상들을 섞은 데이터
   🔴 치명  열이 없습니다: ['사용량']
   🟡 경고  처음 보는 열: ['신규열']
   🟡 경고  '소득' 가 학습 범위를 크게 벗어난 행 1개
   🟡 경고  '도시' 에 처음 보는 값: ['세종']

=> ⭐ **예측을 하기 전에 이걸 잡아야 합니다.**
   📗 원문: 'No data validation -> Silently wrong predictions on bad data'
   '치명'이면 예측을 거부하고, '경고'면 기록하고 계속 — 이게 보통의 정책입니다.


In [20]:
# ✨ 추가 — 7단계: 테스트 (변환기 단위 테스트 + 파이프라인 통합 테스트)
결과 = []

def 검사(이름, 조건, 설명=""):
    결과.append((이름, bool(조건), 설명))

# ── 단위 테스트: 3절에서 만든 변환기 ────────────────────────
t = 표준화변환기().fit(np.array([[1.0, 10.0], [3.0, 20.0], [5.0, 30.0]]))
변환됨 = t.transform(np.array([[3.0, 20.0]]))
검사("표준화: 평균 입력 -> 0", np.allclose(변환됨, 0.0), "평균을 넣으면 0이 나와야 함")
검사("표준화: fit 전 transform 거부",
    (lambda: [False for _ in [표준화변환기().transform(np.ones((1, 2)))]])
    if False else True)
try:
    표준화변환기().transform(np.ones((1, 2)))
    결과[-1] = ("표준화: fit 전 transform 거부", False, "예외가 나야 함")
except RuntimeError:
    pass

c = 중앙값채우기(표시열=True).fit(np.array([[1.0], [np.nan], [3.0]]))
채워짐 = c.transform(np.array([[np.nan]]))
검사("채우기: 빈칸을 중앙값으로", np.isclose(채워짐[0, 0], 2.0), "중앙값 2.0")
검사("채우기: 표시 열이 1", np.isclose(채워짐[0, 1], 1.0), "비어 있었으므로 1")

# ── 통합 테스트: 전체 파이프라인 ────────────────────────────
검사("파이프라인: 예측 길이가 입력과 같다",
    len(전체관.predict(df테스트)) == len(df테스트))
검사("파이프라인: 확률이 [0,1] 안에",
    bool(((전체관.predict_proba(df테스트) >= 0) &
          (전체관.predict_proba(df테스트) <= 1)).all()))
검사("파이프라인: 확률 합이 1",
    np.allclose(전체관.predict_proba(df테스트).sum(axis=1), 1.0))
검사("파이프라인: 한 행만 넣어도 동작",
    len(전체관.predict(df테스트.iloc[:1])) == 1, "실시간 추론은 보통 1행입니다")
검사("파이프라인: 결측이 있어도 동작",
    len(전체관.predict(df테스트.assign(나이=np.nan))) == len(df테스트))
검사("파이프라인: 미지 범주에도 동작", len(전체관.predict(df새)) == len(df새),
    "5절의 handle_unknown='ignore'")
검사("직렬화: 왕복 후 예측이 동일",
    np.array_equal(전체관.predict(df테스트), 불러온것.predict(df테스트)))
검사("재현성: 같은 설정이면 같은 점수",
    abs(Pipeline([("전처리", ColumnTransformer([("숫자", 숫자관, 숫자열),
                                               ("범주", 범주관, 범주열)])),
                  ("모델", LogisticRegression(max_iter=3000))])
        .fit(df훈련, y훈련).score(df테스트, y테스트)
        - 전체관.score(df테스트, y테스트)) < 1e-12)

통과 = sum(1 for _, ok, _ in 결과 if ok)
print(f"{'테스트':<40}{'결과':>8}   비고")
print("-" * 76)
for 이름, ok, 설명 in 결과:
    print(f"{이름:<39}{'✔ 통과' if ok else '✘ 실패':>9}   {설명}")
print("-" * 76)
print(f"{통과}/{len(결과)} 통과")

print(f"\n=> ⭐ 이 테스트들은 **CI에서 매 커밋마다 돌려야** 합니다.")
print("   특히 '한 행만 넣어도 동작'과 '미지 범주에도 동작'이 중요합니다.")
print("   둘 다 **프로덕션에서만 터지는** 종류의 버그거든요.")
print("   노트북에서는 항상 수백 행을 한꺼번에 넣으니 안 드러납니다.")

테스트                                           결과   비고
----------------------------------------------------------------------------
표준화: 평균 입력 -> 0                             ✔ 통과   평균을 넣으면 0이 나와야 함
표준화: fit 전 transform 거부                     ✔ 통과   
채우기: 빈칸을 중앙값으로                              ✔ 통과   중앙값 2.0
채우기: 표시 열이 1                                ✔ 통과   비어 있었으므로 1
파이프라인: 예측 길이가 입력과 같다                        ✔ 통과   
파이프라인: 확률이 [0,1] 안에                         ✔ 통과   
파이프라인: 확률 합이 1                              ✔ 통과   
파이프라인: 한 행만 넣어도 동작                          ✔ 통과   실시간 추론은 보통 1행입니다
파이프라인: 결측이 있어도 동작                           ✔ 통과   
파이프라인: 미지 범주에도 동작                           ✔ 통과   5절의 handle_unknown='ignore'
직렬화: 왕복 후 예측이 동일                            ✔ 통과   
재현성: 같은 설정이면 같은 점수                          ✔ 통과   
----------------------------------------------------------------------------
12/12 통과

=> ⭐ 이 테스트들은 **CI에서 매 커밋마다 돌려야** 합니다.
   특히 '한 행만 넣어도 동작'과 '미지 범주에도 동작'이 중요합니다.
  

---

## 1️⃣1️⃣ 🎒 Ship It — 파이프라인 체크리스트

> 📗 **원문** — 저장소의 `13-ml-pipelines/outputs/prompt-ml-pipeline.md`
> 와 원문의 "Common Pipeline Mistakes" 표를 합쳐 우리말로 옮기고 **절 번호를 붙였습니다.**
>
> 🔗 **이 절은 요약이 아니라 '배포 전 점검표'입니다** — 0~10절이 "각 도구가 무엇인가"였다면,
> 여기는 **모델을 내보내기 전에 하나씩 지워 갈 목록**입니다.

### 🧱 1단계 — 파이프라인 구성

- [ ] 전처리 전부가 **Pipeline 안에** 있나? (파이프라인 밖 변환이 **하나도** 없어야 합니다) — *3·4절*
- [ ] 숫자·범주 열이 **`ColumnTransformer`** 로 분리돼 있나? — *5절*
- [ ] 숫자 열: 결측 채우기 → 스케일링 순서인가? — *5절*
- [ ] 범주 열: 결측 채우기 → 인코딩 순서인가? — *5절*
- [ ] ⭐ **`OneHotEncoder(handle_unknown="ignore")`** 를 썼나? — *5절 (안 쓰면 프로덕션 크래시)*
- [ ] 열 이름이 **config에서** 오나? 코드에 박혀 있지 않나? — *9절*

### 🔒 2단계 — 누출 점검

- [ ] `fit_transform` 을 **훈련 데이터에만** 썼나? — *2·3절*
- [ ] 교차검증을 **파이프라인 객체에** 돌렸나? (전처리된 배열이 아니라) — *6절*
- [ ] **y를 보는 전처리**(타깃 인코딩, 특징 선택, 오버샘플링)가 **파이프라인 안**에 있나? — *2절*
- [ ] 시계열이면 **시간 순서로** 나눴나? — *9단원 0절*
- [ ] 테스트 세트는 **끝까지 봉인**됐나? — *9단원 1절 · 12단원 8절*

### 📦 3단계 — 직렬화와 배포

- [ ] **파이프라인 전체**를 하나의 파일로 저장했나? (모델만 저장 ✘) — *7절*
- [ ] 불러온 뒤 예측이 **원래와 동일**한지 검증했나? — *7절*
- [ ] 저장 파일 안에 **학습된 통계**(평균·중앙값·범주 목록)가 들어 있나? — *7절*
- [ ] 전처리 코드가 **두 벌 존재하지 않나**? (학습용/서빙용 사본 ✘) — *7절*

### 📝 4단계 — 추적과 재현성

- [ ] 실행마다 **파라미터·지표·모델**을 기록하나? — *8절*
- [ ] **난수를 쓰는 모든 객체**에 `random_state` 가 있나? — *9절 (가장 많이 빼먹습니다)*
- [ ] 의존성 버전을 **고정**했나? — *9절*
- [ ] **데이터 버전**(해시 또는 DVC)을 기록하나? — *9절*
- [ ] 하이퍼파라미터가 **config 파일**에 있나? — *9절*
- [ ] **git 커밋 해시**가 실행 기록에 남나? — *8절*

### 🛡️ 5단계 — 운영 안전장치

- [ ] 예측 전에 **스키마 검증**을 하나? — *10절*
- [ ] **처음 보는 범주 값**을 모니터링하나? — *5·10절*
- [ ] **결측률 급증**을 감지하나? — *10절*
- [ ] **한 행 입력**으로 테스트했나? (실시간 추론은 1행입니다) — *10절*
- [ ] 변환기 **단위 테스트**와 파이프라인 **통합 테스트**가 CI에 있나? — *10절*
- [ ] 이전 버전으로 **롤백**할 수 있나? — *8절 모델 레지스트리*

---

### ⚠️ 📗 가장 많이 하는 실수 여섯 가지

| # | 실수 | 증상 | 해법 |
| --- | --- | --- | --- |
| 1 | 나누기 전에 전체로 fit | 검증 점수가 부풀려짐. **에러 없음** | Pipeline + `cross_val_score` |
| 2 | 특징 공학을 파이프라인 밖에 | 학습·서빙 변환이 갈라짐 | 전부 Pipeline 안에 |
| 3 | 미지 범주 미처리 | **프로덕션 크래시** | `handle_unknown="ignore"` |
| 4 | 열 이름 하드코딩 | 스키마 변경 시 깨짐 | config의 열 목록 |
| 5 | 데이터 검증 없음 | **조용히 틀린 예측** | 예측 전 스키마 검사 |
| 6 | 학습/서빙 불일치 | 재현 불가, 원인 추적 난항 | **하나의 Pipeline 객체** |

> 🔑 **1·5·6번의 공통점을 보세요 — 전부 "에러가 안 납니다."**
> 그래서 체크리스트가 필요한 겁니다. 터지는 버그는 알아서 발견되지만,
> **조용한 버그는 찾아 나서야만 발견됩니다.**

---

## 1️⃣2️⃣ 📝 자가 점검 퀴즈

> 📗 **원문** — 저장소의 `13-ml-pipelines/quiz.json`
> 우리말로 옮기고, 각 문항이 이 노트북 **몇 절**에 있는지 표시했습니다.
>
> 💡 클론한 저장소가 있으면 Claude Code에서 `/check-understanding 2` 로도 풀 수 있습니다.

**먼저 답을 정한 뒤에** 아래 코드 셀을 실행해 정답을 확인하세요.

### 사전 점검 — 배우기 전에 알았어야 하는 것

**1. ML 파이프라인에서 데이터 누출이란?**

1. 인코딩 중에 특징이 누락되는 것
2. 모델이 데이터를 처리하기에 너무 느린 것
3. 테스트 세트나 미래 데이터의 정보가 학습 과정을 오염시키는 것
4. 전처리 중에 데이터가 실수로 삭제되는 것

**2. 훈련/테스트로 나누기 전에 스케일러를 전체 데이터로 fit 하는 게 왜 누출인가요?**

1. 스케일러의 통계(평균, 표준편차)에 테스트 데이터가 포함되어, 모델이 간접적으로 테스트 정보를 봅니다
2. 스케일링이 데이터 분포를 바꿉니다
3. 스케일링은 테스트 세트에만 해야 합니다
4. 큰 데이터에서 스케일러 fit 이 너무 오래 걸립니다

### 사후 점검 — 이 단원을 배운 뒤에 알아야 하는 것

**3. sklearn에서 파이프라인 단계의 `fit_transform` 과 `transform` 의 차이는?**

1. 같습니다. 둘 다 fit 하고 transform 합니다
2. `fit_transform` 은 데이터에서 파라미터를 배우고 변환까지 하고, `transform` 은 이미 배운 파라미터만 적용합니다
3. `transform` 은 훈련 데이터용이고 `fit_transform` 은 테스트 데이터용입니다
4. `fit_transform` 이 더 느리지만 더 정확합니다

**4. 실제 데이터셋에 `ColumnTransformer` 가 왜 필요한가요?**

1. 파이프라인을 여러 CPU에서 병렬 실행하게 해 줍니다
2. 모든 열을 같은 자료형으로 변환합니다
3. 같은 파이프라인 안에서 **숫자 열과 범주 열에 다른 전처리**를 적용합니다
4. 결측값이 있는 열을 자동으로 제거합니다

**5. 프로덕션 모델이 학습 중 본 적 없는 범주 값('새 카테고리')을 받았습니다. 파이프라인은 무엇을 해야 하나요?**

1. 미지 범주를 숫자 0으로 변환
2. 미지 범주를 우아하게 처리 — 예를 들어 'unknown' 버킷을 쓰거나, 타깃 인코딩이면 전체 평균을 쓰기
3. 해당 행을 아예 무시하고 예측을 내지 않기
4. 전체 모델을 처음부터 재학습

In [21]:
# 📝 정답과 해설 (+ 이 노트북 어디서 다뤘는지)
퀴즈 = [
    ("1. 데이터 누출이란", 3,
     "데이터 누출은 예측 시점에 없을 정보를 학습에 쓰는 것입니다. 예: 테스트 데이터까지 "
     "포함해 스케일러의 평균·표준편차를 계산하는 것.",
     "2절 (전처리 종류별로 누출 크기 측정) · 6절 (교차검증에서) · 9단원 2절"),
    ("2. 나누기 전에 스케일러를 fit 하면", 1,
     "스케일러를 전체 데이터로 fit 하면 평균과 표준편차에 테스트 샘플 정보가 인코딩됩니다. "
     "훈련 특징이 테스트 통계로 변환되므로 미래 정보가 학습에 새어 듭니다.",
     "2절 — 전처리 종류별로 누출 크기를 측정했습니다. "
     "⚠️ 다만 방향이 일정하지 않습니다 — MinMaxScaler 는 점수를 45%p 망가뜨렸습니다"),
    ("3. fit_transform vs transform", 2,
     "fit_transform 은 데이터에서 통계(평균, 표준편차 등)를 계산하고 변환까지 합니다. "
     "transform 은 이미 배운 통계를 적용만 합니다. 테스트 데이터에는 누출을 피하려면 "
     "transform 을 써야 합니다.",
     "3절 (직접 만들어 두 메서드를 분리) · 4절 (Pipeline 내부 호출을 추적해 로그로 확인)"),
    ("4. ColumnTransformer 가 필요한 이유", 3,
     "실제 데이터는 자료형이 섞여 있습니다. 숫자 열은 스케일링이, 범주 열은 인코딩이 "
     "필요합니다. ColumnTransformer 는 하나의 파이프라인 안에서 각 열 묶음을 "
     "적절한 변환기로 보냅니다.",
     "5절 — DataFrame 을 그대로 넣고 열 이름으로 라우팅하는 걸 확인했습니다"),
    ("5. 처음 보는 범주 값이 오면", 2,
     "견고한 파이프라인은 프로덕션에서 미지 범주를 예상해야 합니다. 원핫에는 'unknown' "
     "대체, 타깃 인코딩에는 전체 평균 기본값 같은 해법이 있습니다.",
     "5절 — handle_unknown='ignore' 는 0 벡터를 내고 계속 동작, "
     "'error' 는 예외로 터지는 걸 둘 다 재현했습니다 · 10절 (모니터링)"),
]

print("=" * 80)
for 제목, 정답, 해설, 어디 in 퀴즈:
    print(f"{제목}")
    print(f"   정답: {정답}번")
    print(f"   해설: {해설}")
    print(f"   이 노트북: {어디}")
    print("-" * 80)

print("\n💡 채점 기준")
print("   5개 다 맞음 -> Phase 2 완주입니다! 14단원(나이브 베이즈)으로 가거나")
print("                 Phase 3로 넘어가세요")
print("   3~4개 맞음  -> 틀린 문항의 '이 노트북' 절만 다시 읽으세요")
print("   2개 이하    -> 2·3절(누출과 fit/transform 분리)을 먼저 다시 보세요.")
print("                 이 단원의 나머지가 다 거기서 나옵니다")

1. 데이터 누출이란
   정답: 3번
   해설: 데이터 누출은 예측 시점에 없을 정보를 학습에 쓰는 것입니다. 예: 테스트 데이터까지 포함해 스케일러의 평균·표준편차를 계산하는 것.
   이 노트북: 2절 (전처리 종류별로 누출 크기 측정) · 6절 (교차검증에서) · 9단원 2절
--------------------------------------------------------------------------------
2. 나누기 전에 스케일러를 fit 하면
   정답: 1번
   해설: 스케일러를 전체 데이터로 fit 하면 평균과 표준편차에 테스트 샘플 정보가 인코딩됩니다. 훈련 특징이 테스트 통계로 변환되므로 미래 정보가 학습에 새어 듭니다.
   이 노트북: 2절 — 전처리 종류별로 누출 크기를 측정했습니다. ⚠️ 다만 방향이 일정하지 않습니다 — MinMaxScaler 는 점수를 45%p 망가뜨렸습니다
--------------------------------------------------------------------------------
3. fit_transform vs transform
   정답: 2번
   해설: fit_transform 은 데이터에서 통계(평균, 표준편차 등)를 계산하고 변환까지 합니다. transform 은 이미 배운 통계를 적용만 합니다. 테스트 데이터에는 누출을 피하려면 transform 을 써야 합니다.
   이 노트북: 3절 (직접 만들어 두 메서드를 분리) · 4절 (Pipeline 내부 호출을 추적해 로그로 확인)
--------------------------------------------------------------------------------
4. ColumnTransformer 가 필요한 이유
   정답: 3번
   해설: 실제 데이터는 자료형이 섞여 있습니다. 숫자 열은 스케일링이, 범주 열은 인코딩이 필요합니다. ColumnTransformer 는 하나의 파이프라인 안에서 각 열 

---

## 1️⃣3️⃣ 용어 사전

> 📗 **원문** — "Key Terms" (설명은 쉬운 말로 다시 썼습니다)

| 용어 | 흔히 하는 말 | 진짜 무슨 뜻인가 |
| --- | --- | --- |
| **파이프라인** | "변환 + 모델 묶음" | 순서가 있는 변환기들과 모델을 **하나의 단위로** 적용해 누출을 막는 객체 |
| **데이터 누출** | "테스트 정보가 샜다" | 훈련 세트 밖의 정보를 모델 구축에 쓰는 것. **성능 추정이 부풀려집니다** |
| **`fit_transform` vs `transform`** | "둘이 뭐가 달라" | 앞은 **통계를 배우고** 변환, 뒤는 **배운 통계로** 변환만. 이 분리가 파이프라인의 핵심 |
| **ColumnTransformer** | "열마다 다른 전처리" | 열 묶음마다 다른 파이프라인을 적용하고 결과를 합칩니다 |
| **`handle_unknown="ignore"`** | "모르는 값 무시" | 처음 보는 범주에 0 벡터를 내놓아 **크래시를 막습니다.** 프로덕션 필수 |
| **실험 추적** | "실행 기록 남기기" | 모든 학습 실행의 파라미터·지표·아티팩트·코드 버전을 기록 |
| **MLflow** | "추적하고 배포하기" | 실험 추적·모델 레지스트리·배포를 지원하는 오픈소스 플랫폼 |
| **DVC** | "데이터용 git" | 큰 데이터 파일의 버전 관리. **해시는 git에, 데이터는 원격 저장소에** |
| **모델 레지스트리** | "모델 버전 목록" | 버전과 단계(staging/production/archived)를 추적하는 시스템 |
| **학습/서빙 불일치** | "노트북에선 됐는데" | 학습 때와 추론 때 데이터 처리가 달라져 **조용히 틀린 결과**가 나오는 것 |
| **재현성** | "같은 코드, 같은 결과" | 같은 코드·데이터·설정으로 **동일한 결과**를 얻는 능력 |
| **데이터 검증** | "스키마 검사" | 예측 전에 열·자료형·범위·결측률이 학습 때와 맞는지 확인 |

---

## ✅ 이 단원 요약 (7줄)

1. **프로덕션 장애 네 가지**(누출·파라미터 미저장·사본 분기·미지 범주)는 전부 **파이프라인으로 해결**됩니다.
2. 파이프라인의 핵심은 **`fit_transform`(훈련)과 `transform`(추론)의 분리** 하나입니다. 3절에서 40줄로 만들었습니다.
3. ⚠️ **누출은 y를 보는 전처리에서 가장 큽니다.** 하지만 전부 파이프라인에 넣으면 **구분해 고민할 필요가 없어집니다**(2절).
4. **`ColumnTransformer`** 가 숫자·범주를 라우팅합니다. ⭐ **`handle_unknown="ignore"`** 없으면 프로덕션이 멈춥니다(5절).
5. **교차검증은 파이프라인 객체에** 돌리세요. 그러면 **겹마다 전처리가 새로 fit** 됩니다(6절, fit 호출 5회를 로그로 확인).
6. **파이프라인 전체를 하나로 저장**하세요. 전처리 통계가 같이 들어갑니다. 코드가 두 벌이면 **조용히 갈라집니다**(7절).
7. **재현성 네 조건**: 시드·의존성·데이터 버전·설정 파일. ⚠️ `random_state` 를 가장 많이 빼먹습니다(9절).

> 💡 그리고 이 단원 전체의 교훈: **"에러가 안 나는 버그가 가장 위험합니다."**
> 누출도, 학습/서빙 불일치도, 나쁜 데이터도 **전부 조용히** 틀립니다.
> 그래서 11절의 체크리스트가 필요한 겁니다.

---

## 📝 연습문제

> 📗 **원문** — "Exercises" (전부 본문에서 풀었습니다)

1. 숫자 3개 + 범주 2개 데이터에 `ColumnTransformer` 파이프라인을 만들고 5-겹 교차검증으로 학습하세요. → *5·6절에서 함*
2. 의도적으로 누출을 만들어(나누기 전 전체로 스케일러 fit) 깨끗한 파이프라인과 교차검증 점수를 비교하세요. 차이가 얼마나 큰가요? → *2·6절에서 함*
3. `joblib.dump` 로 직렬화하고 별도 스크립트에서 불러 예측이 동일한지 검증하세요. → *7절에서 함*
4. 다항 특징(2차)을 만드는 커스텀 변환기를 추가하세요. 파이프라인의 **어디에** 넣어야 하나요? → *아래 도전 과제 6번*
5. MLflow 추적을 설정하고 5개 실험을 돌려 비교하세요. → *8절에서 함*

### ✨ 추가 도전 과제

6. 연습문제 4번을 실제로 해 보세요. 다항 특징 변환기를 **스케일링 전**에 넣으면? **후에** 넣으면? 결과가 다른가요? (힌트: $x^2$ 와 $(x_\text{scaled})^2$ 는 다릅니다)
7. 10절의 `데이터검증기` 에 **분포 이동 감지**를 추가하세요. KS 검정이나 평균·분산 비교로요. 2절의 분포 이동 실험 데이터로 테스트하세요.
8. 7절의 파이프라인을 **FastAPI** 로 감싸 `/predict` 엔드포인트를 만드세요. 한 행 JSON을 받아 확률을 돌려주면 됩니다. 10절 테스트의 '한 행 입력'이 왜 중요한지 체감할 수 있습니다.
9. 8단원 3절의 **타깃 인코더**를 sklearn 규약(`BaseEstimator`, `TransformerMixin`)에 맞게 다시 만들어 파이프라인에 넣으세요. 6절 교차검증에서 누출이 정말 사라지나요?
10. 12단원의 **Optuna** 와 이 단원의 파이프라인을 합치세요. `전처리__숫자__채우기__strategy` 같은 전처리 하이퍼파라미터까지 탐색 대상에 넣으면, 모델만 튜닝한 것보다 나은가요?

---

## 🎓 Phase 2를 마치며

> ✨ **추가** — 13단원까지 오셨습니다. 1~13단원이 어떻게 이어졌는지 되짚어 봅니다.

| 묶음 | 단원 | 배운 것 |
| --- | --- | --- |
| **모델** | 1~7 | 선형·로지스틱·트리·SVM·KNN·군집화 — **알고리즘들** |
| **데이터** | 8 | 특징 공학 — **"모델보다 데이터가 중요하다"** |
| **평가** | 9·10 | 평가 지표와 편향-분산 — **"제대로 재는 법"** |
| **개선** | 11·12 | 앙상블과 튜닝 — **"더 좋게 만드는 법"** |
| **공정** | 13 | 파이프라인 — **"무너지지 않게 만드는 법"** |

> 💡 그리고 Phase 2 전체를 관통한 교훈 하나:
> **"통념 대신 재서 결정하세요."**
>
> 이 노트북들을 만들면서 원문의 주장을 여러 번 측정했고, 그중 **여덟 개가 조건부이거나 틀렸습니다.**
> 로그 변환도, TF-IDF도, 특징 선택도, 이중 하강도, XGBoost 대 신경망도,
> 다수결 확률도, 배깅도, 격자 대 무작위도 — **데이터에 따라 결과가 달랐습니다.**
>
> 책에 적힌 걸 외우는 게 아니라 **여러분 데이터로 재 보는 것**이 이 과정의 목표입니다.

---

## 📚 더 읽을거리

> 📗 **원문** — "Further Reading"

- [scikit-learn Pipeline 문서](https://scikit-learn.org/stable/modules/compose.html) — 공식 참조
- [MLflow 문서](https://mlflow.org/docs/latest/index.html) — 실험 추적과 모델 레지스트리
- [DVC 문서](https://dvc.org/doc) — 데이터 버전 관리
- [Sculley et al., Hidden Technical Debt in Machine Learning Systems (2015)](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html) — **ML 시스템 복잡도의 고전 논문**
- [Google ML Best Practices: Rules of ML](https://developers.google.com/machine-learning/guides/rules-of-ml) — 실전 프로덕션 ML 조언

In [22]:
# ── 정리 — 이 노트북이 만든 임시 파일들을 확인하고 지웁니다 ──────────
만든파일 = []
for 뿌리, _, 파일들 in os.walk(작업방):
    for f in 파일들:
        만든파일.append(os.path.relpath(os.path.join(뿌리, f), 작업방))

print(f"임시 작업 폴더: {작업방}")
print(f"만들어진 파일 {len(만든파일)}개 (앞 10개)")
for f in sorted(만든파일)[:10]:
    print(f"   {f}")
if len(만든파일) > 10:
    print(f"   ... 외 {len(만든파일)-10}개 (대부분 MLflow 아티팩트)")

print(f"\n💡 직접 둘러보려면 아래 줄을 주석 처리하고 실행하세요.")
print(f"   터미널에서: mlflow ui --backend-store-uri sqlite:///{추적DB}")

import shutil
shutil.rmtree(작업방, ignore_errors=True)
print(f"\n정리 완료 — 임시 폴더를 지웠습니다 (존재 여부: {os.path.exists(작업방)})")
print("\n🎓 **Phase 2 · 13단원 완주입니다. 수고하셨습니다!**")

임시 작업 폴더: /var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2
만들어진 파일 4개 (앞 10개)
   churn-pipeline.joblib
   config.json
   mlflow.db
   requirements-lock.txt

💡 직접 둘러보려면 아래 줄을 주석 처리하고 실행하세요.
   터미널에서: mlflow ui --backend-store-uri sqlite:////var/folders/9w/m7__0lhd16xcy6bzy6qwytpc0000gn/T/pipeline-13-gdrf_pt2/mlflow.db

정리 완료 — 임시 폴더를 지웠습니다 (존재 여부: False)

🎓 **Phase 2 · 13단원 완주입니다. 수고하셨습니다!**
